In [ ]:
import os
import time
import pandas as pd

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.common.action_chains import ActionChains

from selenium.webdriver.common.keys import Keys

from selenium.common.exceptions import (

    TimeoutException,

    StaleElementReferenceException,

    NoSuchElementException,

    WebDriverException

)

# ============================================================

# CREDENCIAIS

# ============================================================

CPF = "SUA_SENHA"
SENHA = "SUA_SENHA"


# ============================================================

# URL

# ============================================================



URL = "https://minhaarea.sp.gov.br/plataformasp"

df = pd.read_excel(r"X:\GOB\Teste Automação - NÃO DELETAR\PLANILHA TESTE.xlsx", dtype={"Processo SEI": str, "NFS": str, "CONTRATO": str})





# ============================================================

# DATAFRAME

# ============================================================



if "df" not in globals():



    raise Exception(

        "O DataFrame 'df' não existe."

    )





if "Processo SEI" not in df.columns:



    raise Exception(

        "A coluna 'Processo SEI' não foi encontrada no DataFrame."

    )





if "Data Emissão NF" not in df.columns:



    raise Exception(

        "A coluna 'Data Emissão NF' não foi encontrada no DataFrame."

    )





if "NFS" not in df.columns:



    raise Exception(

        "A coluna 'NFS' não foi encontrada no DataFrame."

    )





# ============================================================

# PERFIL PERSISTENTE

# ============================================================



PERFIL_CHROME = r"C:\AutomacaoChrome\UserData"



os.makedirs(

    PERFIL_CHROME,

    exist_ok=True

)





# ============================================================

# CHROME

# ============================================================



options = webdriver.ChromeOptions()



options.add_argument(

    f"--user-data-dir={PERFIL_CHROME}"

)



options.add_argument(

    "--profile-directory=Default"

)





driver = webdriver.Chrome(

    options=options

)



driver.maximize_window()



driver.get(

    URL

)



wait = WebDriverWait(

    driver,

    30

)





# ============================================================

# 1. CLICA NO LINK INICIAL

# ============================================================



xpath_inicial = (

    "/html/body/div/div/section/main/div[2]/div/"

    "sp-page-row/div/div/span[3]/div/div/div/div/"

    "div/div/div/div/div/a"

)





elemento = wait.until(

    EC.element_to_be_clickable(

        (

            By.XPATH,

            xpath_inicial

        )

    )

)





elemento.click()





print(

    "1 - Página de login / autenticação aberta"

)





# ============================================================

# 2. VERIFICA LOGIN

# ============================================================



xpath_cpf = (

    "/html/body/div[1]/main/form/div[1]/div[2]/input"

)





print(

    "Verificando se existe sessão autenticada..."

)





try:



    campo_cpf = WebDriverWait(

        driver,

        15

    ).until(

        EC.element_to_be_clickable(

            (

                By.XPATH,

                xpath_cpf

            )

        )

    )



    login_necessario = True



    print(

        "Sessão não encontrada."

    )



    print(

        "Login será realizado."

    )





except TimeoutException:



    login_necessario = False



    print(

        "Sessão autenticada encontrada."

    )



    print(

        "CPF e senha serão ignorados."

    )





# ============================================================

# 3. LOGIN

# ============================================================



if login_necessario:

    if not CPF or not SENHA:
        raise RuntimeError("Defina SEI_CPF e SEI_SENHA nas variáveis de ambiente antes de iniciar.")



    campo_cpf.click()



    campo_cpf.clear()



    campo_cpf.send_keys(

        CPF

    )



    print(

        "2 - CPF preenchido"

    )





    botao_continuar = wait.until(

        EC.element_to_be_clickable(

            (

                By.XPATH,

                "/html/body/div[1]/main/form/div[1]/div[2]/div/button"

            )

        )

    )



    botao_continuar.click()



    print(

        "3 - Botão continuar clicado"

    )





    campo_senha = wait.until(

        EC.element_to_be_clickable(

            (

                By.XPATH,

                "/html/body/div[1]/main/form/div[1]/div[1]/input"

            )

        )

    )



    campo_senha.click()



    campo_senha.clear()



    campo_senha.send_keys(

        SENHA

    )



    print(

        "4 - Senha preenchida"

    )





    botao_enter = wait.until(

        EC.element_to_be_clickable(

            (

                By.XPATH,

                "/html/body/div[1]/main/form/div[1]/div[2]/div/button[2]"

            )

        )

    )



    botao_enter.click()



    print(

        "5 - Login enviado"

    )





else:



    print(

        "2 - CPF ignorado"

    )



    print(

        "3 - Continuar ignorado"

    )



    print(

        "4 - Senha ignorada"

    )



    print(

        "5 - Login ignorado"

    )





# ============================================================

# 6. AGUARDA

# ============================================================



print(

    "6 - Aguardando 20 segundos para carregar a aplicação..."

)



time.sleep(

    20

)





# ============================================================

# 7. PROCURA NOW-CARD

# ============================================================



print(

    "7 - Procurando o elemento..."

)





script = """

function buscarElemento(root) {



    let elementos;



    try {

        elementos = root.querySelectorAll('*');

    } catch (e) {

        return null;

    }



    for (const el of elementos) {



        if (

            el.tagName &&

            el.tagName.toLowerCase() === 'now-card'

        ) {



            let rootCard = el.shadowRoot || el;



            let div = rootCard.querySelector('div');



            if (div) {

                return div;

            }



            return el;

        }



        if (el.shadowRoot) {



            const encontrado = buscarElemento(el.shadowRoot);



            if (encontrado) {

                return encontrado;

            }

        }

    }



    return null;

}



return buscarElemento(document);

"""





elemento_final = driver.execute_script(

    script

)





# ============================================================

# 8. NOW-CARD

# ============================================================



if elemento_final:



    print(

        "8 - NOW-CARD encontrado"

    )





    div_final = driver.execute_script("""

        const card = arguments[0];



        const root = card.shadowRoot || card;



        return root.querySelector('div');

    """, elemento_final)





    if div_final:



        print(

            "9 - DIV interna encontrada"

        )





        driver.execute_script(

            "arguments[0].scrollIntoView({block:'center'});",

            div_final

        )



        time.sleep(

            1

        )





        driver.execute_script(

            "arguments[0].click();",

            div_final

        )





        print(

            "10 - CLIQUE FINAL REALIZADO COM SUCESSO!"

        )





    else:



        raise Exception(

            "DIV interna não encontrada."

        )





else:



    raise Exception(

        "NOW-CARD não encontrado."

    )





# ============================================================

# 11. AGUARDA SEI

# ============================================================



print("")

print(

    "11 - Aguardando janela / aba do SEI..."

)



time.sleep(

    3

)





# ============================================================

# 12. LOCALIZA ABA DO SEI

# ============================================================



janela_sei = None



inicio = time.time()





while time.time() - inicio < 30:



    for handle in driver.window_handles:



        try:



            driver.switch_to.window(

                handle

            )



            driver.switch_to.default_content()





            print(

                "Verificando:",

                driver.title,

                "|",

                driver.current_url

            )





            if (

                "sei" in driver.current_url.lower()

                or

                "SEI" in driver.title

            ):



                janela_sei = handle



                break





        except Exception:



            continue





    if janela_sei:



        break





    time.sleep(

        1

    )





if not janela_sei:



    raise Exception(

        "Não foi possível localizar a aba do SEI."

    )





# ============================================================

# 13. GARANTE ABA SEI

# ============================================================



driver.switch_to.window(

    janela_sei

)



driver.switch_to.default_content()





print("")

print(

    "12 - Aba do SEI localizada."

)



print(

    "Título:",

    driver.title

)



print(

    "URL:",

    driver.current_url

)





# ============================================================

# 14. FECHA MODAL

# ============================================================



print("")

print(

    "13 - Verificando janela de novidades..."

)





inicio_modal = time.time()



modal_fechado = False





while time.time() - inicio_modal < 10:



    try:



        driver.switch_to.default_content()





        resultado_modal = driver.execute_script("""

            const botao = document.querySelector(

                "div.sparkling-modal-close a"

            );



            if (!botao) {

                return false;

            }



            botao.click();



            return true;

        """)





        if resultado_modal:



            modal_fechado = True



            print(

                "13 - Janela de novidades fechada."

            )



            break





    except Exception:



        pass





    time.sleep(

        0.5

    )





if not modal_fechado:



    print(

        "13 - Modal não encontrado."

    )






# ============================================================
# PROCESSAMENTO: UM FLUXO COMPLETO POR LINHA
# Autenticacao, NOW-CARD e abertura do SEI acima: somente uma vez.
# Nenhuma etapa do fluxo entre Pesquisa Rapida e INCLUIR e omitida.
# ============================================================
import csv
from datetime import datetime
from pathlib import Path

ARQUIVO_LOG = Path(r"X:\GOB\Teste Automação - NÃO DELETAR\log_automacao_sei.csv")
COLUNAS_OBRIGATORIAS = ["Processo SEI", "Data Emissão NF", "NFS", "CONTRATO", "Nota Fiscal Eletrônica", "AP"]
ausentes = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if ausentes:
    raise ValueError(f"Colunas ausentes no DataFrame: {ausentes}")

# A propria janela do SEI e reutilizada; nao abre URL nem repete login.
JANELA_SEI_PRINCIPAL = janela_sei

# Acrescenta AP sem perder os registros anteriores do log existente.
COLUNAS_LOG = ["data_hora", "linha_excel", "processo_sei", "AP", "status", "detalhe"]

def preparar_log():
    ARQUIVO_LOG.parent.mkdir(parents=True, exist_ok=True)
    if not ARQUIVO_LOG.exists() or ARQUIVO_LOG.stat().st_size == 0:
        return
    with ARQUIVO_LOG.open("r", newline="", encoding="utf-8-sig") as arquivo:
        leitor = csv.DictReader(arquivo, delimiter=";")
        colunas_atuais = leitor.fieldnames or []
        if colunas_atuais == COLUNAS_LOG:
            return
        if not colunas_atuais or any(c not in COLUNAS_LOG for c in colunas_atuais):
            raise ValueError(f"Cabecalho inesperado no log: {colunas_atuais}")
        linhas_anteriores = list(leitor)
    # Backup antes de migrar o log antigo (sem coluna AP).
    from shutil import copy2
    backup = ARQUIVO_LOG.with_name(ARQUIVO_LOG.stem + "_backup_antes_AP" + ARQUIVO_LOG.suffix)
    if not backup.exists():
        copy2(ARQUIVO_LOG, backup)
    with ARQUIVO_LOG.open("w", newline="", encoding="utf-8-sig") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=COLUNAS_LOG, delimiter=";")
        escritor.writeheader()
        for registro in linhas_anteriores:
            escritor.writerow({coluna: registro.get(coluna, "") for coluna in COLUNAS_LOG})

def escrever_log(linha_excel, processo, ap, status, detalhe):
    ARQUIVO_LOG.parent.mkdir(parents=True, exist_ok=True)
    criar_cabecalho = not ARQUIVO_LOG.exists() or ARQUIVO_LOG.stat().st_size == 0
    with ARQUIVO_LOG.open("a", newline="", encoding="utf-8-sig") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=COLUNAS_LOG, delimiter=";")
        if criar_cabecalho:
            escritor.writeheader()
        escritor.writerow({
            "data_hora": datetime.now().isoformat(timespec="seconds"),
            "linha_excel": linha_excel,
            "processo_sei": processo,
            "AP": ap,
            "status": status,
            "detalhe": detalhe,
        })

preparar_log()

TEMPO_ESPERA_ENTRE_PROCESSOS = 20  # segundos apos o clique INCLUIR
TEMPO_MAXIMO_CAMPO_EXTERNO = 75  # segundos para a tela do tipo Externo aparecer

TOTAL = len(df)
for numero_linha, (indice_linha_atual, dados_linha) in enumerate(df.iterrows(), start=1):
    # Excel: cabecalho na linha 1, dados comecam na linha 2.
    linha_excel = numero_linha + 1
    valor_processo_linha = dados_linha.get("Processo SEI")
    processo_log = "" if pd.isna(valor_processo_linha) else str(valor_processo_linha).strip()
    valor_ap_log = dados_linha.get("AP")
    ap_log = "" if pd.isna(valor_ap_log) else str(valor_ap_log).strip()
    if ap_log.endswith(".0") and ap_log[:-2].isdigit():
        ap_log = ap_log[:-2]
    print("\n" + "=" * 75)
    print(f"LINHA {numero_linha}/{TOTAL} | Excel {linha_excel} | Processo SEI: {processo_log}")
    print("=" * 75)

    # Campos indispensaveis: linhas sem valores nao sao cadastradas.
    colunas_validar = ["Processo SEI", "Data Emissão NF", "NFS", "CONTRATO", "Nota Fiscal Eletrônica", "AP"]
    vazias = [c for c in colunas_validar if pd.isna(dados_linha[c]) or not str(dados_linha[c]).strip()]
    if vazias:
        motivo = "Campos vazios: " + ", ".join(vazias)
        print("IGNORADA:", motivo)
        escrever_log(linha_excel, processo_log, ap_log, "IGNORADA", motivo)
        continue

    try:
        driver.switch_to.window(JANELA_SEI_PRINCIPAL)
        driver.switch_to.default_content()
        WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.ID, "txtPesquisaRapida"))
        )
        # Todo o codigo original desde obter Processo SEI ate clicar INCLUIR:
        valor_processo = df.loc[

            indice_linha_atual,

            "Processo SEI"

        ]





        if (

            isinstance(

                valor_processo,

                float

            )

            and

            valor_processo.is_integer()

        ):



            processo_sei = str(

                int(

                    valor_processo

                )

            )



        else:



            processo_sei = str(

                valor_processo

            ).strip()





        print("")

        print(

            "15 - Processo SEI:"

        )



        print(

            processo_sei

        )





        # ============================================================

        # 16. PESQUISA RÁPIDA

        # ============================================================



        driver.switch_to.default_content()





        campo_processo = WebDriverWait(

            driver,

            30

        ).until(

            EC.element_to_be_clickable(

                (

                    By.ID,

                    "txtPesquisaRapida"

                )

            )

        )





        print(

            "16 - Campo Pesquisa Rápida encontrado."

        )





        # ============================================================

        # 17. PREENCHE PROCESSO

        # ============================================================



        campo_processo.click()



        campo_processo.clear()



        campo_processo.send_keys(

            processo_sei

        )





        print(

            "17 - Processo inserido."

        )





        # ============================================================

        # 18. LUPA

        # ============================================================



        xpath_pesquisar = (

            "//img[@title='Pesquisa Rápida']"

        )





        botao_pesquisar = WebDriverWait(

            driver,

            30

        ).until(

            EC.element_to_be_clickable(

                (

                    By.XPATH,

                    xpath_pesquisar

                )

            )

        )





        # ============================================================

        # 19. CLICA NA LUPA

        # ============================================================



        try:



            botao_pesquisar.click()





        except Exception:



            driver.execute_script(

                """

                const botao =

                    document.querySelector(

                        "img[title='Pesquisa Rápida']"

                    );



                if (botao) {

                    botao.click();

                }

                """

            )





        print("")

        print(

            "19 - PESQUISA REALIZADA!"

        )





        # ============================================================

        # 20. AGUARDA PROCESSO

        # ============================================================



        print("")

        print(

            "20 - Aguardando abertura do processo..."

        )





        driver.switch_to.default_content()





        WebDriverWait(

            driver,

            30

        ).until(

            EC.presence_of_element_located(

                (

                    By.ID,

                    "ifrArvore"

                )

            )

        )





        print(

            "20 - Página do processo carregada."

        )





        # ============================================================

        # 21. INCLUIR DOCUMENTO

        # ============================================================



        print("")

        print(

            "21 - Procurando Incluir Documento..."

        )





        script_incluir_documento = """

        function procurar(win, caminho) {



            let doc;



            try {



                doc = win.document;



            } catch (erro) {



                return {

                    encontrou: false

                };

            }





            let elemento = null;

            let metodo = null;





            elemento = doc.querySelector(

                "img[title='Incluir Documento']"

            );



            if (elemento) {

                metodo = "TITLE";

            }





            if (!elemento) {



                elemento = doc.querySelector(

                    "img[alt='Incluir Documento']"

                );



                if (elemento) {

                    metodo = "ALT";

                }

            }





            if (!elemento) {



                elemento = doc.querySelector(

                    "img[src*='documento_incluir']"

                );



                if (elemento) {

                    metodo = "SRC";

                }

            }





            if (!elemento) {



                elemento = doc.querySelector(

                    "a[onclick*='documento_gerar_multiplo']"

                );



                if (elemento) {

                    metodo = "ONCLICK";

                }

            }





            if (!elemento) {



                elemento = doc.querySelector(

                    "a[onclick*='documento_gerar']"

                );



                if (elemento) {

                    metodo = "ONCLICK documento_gerar";

                }

            }





            if (!elemento) {



                try {



                    elemento = doc.evaluate(

                        "/html/body/div[1]/div/div/div[2]/div/a[1]/img",

                        doc,

                        null,

                        XPathResult.FIRST_ORDERED_NODE_TYPE,

                        null

                    ).singleNodeValue;





                    if (elemento) {

                        metodo = "XPATH";

                    }



                } catch (erro) {



                }

            }





            if (elemento) {



                let alvo = elemento;





                if (

                    elemento.tagName &&

                    elemento.tagName.toLowerCase() === "img"

                ) {



                    const link =

                        elemento.closest("a");





                    if (link) {

                        alvo = link;

                    }

                }





                try {



                    alvo.scrollIntoView({

                        block: "center",

                        inline: "center"

                    });



                } catch (erro) {



                }





                alvo.click();





                return {



                    encontrou: true,



                    clicou: true,



                    caminho: caminho,



                    metodo: metodo



                };

            }





            for (

                let i = 0;

                i < win.frames.length;

                i++

            ) {



                try {



                    const nomeFrame =

                        win.frames[i].name ||

                        ("frame[" + i + "]");





                    const resultado = procurar(

                        win.frames[i],

                        caminho + " -> " + nomeFrame

                    );





                    if (

                        resultado &&

                        resultado.encontrou

                    ) {



                        return resultado;

                    }





                } catch (erro) {



                }

            }





            return {

                encontrou: false

            };

        }





        return procurar(

            window,

            "DOCUMENTO PRINCIPAL"

        );

        """





        clicou_incluir = False



        inicio = time.time()





        while time.time() - inicio < 60:



            try:



                driver.switch_to.default_content()





                resultado_incluir = driver.execute_script(

                    script_incluir_documento

                )





                if (

                    resultado_incluir

                    and

                    resultado_incluir.get(

                        "encontrou"

                    )

                    and

                    resultado_incluir.get(

                        "clicou"

                    )

                ):



                    clicou_incluir = True





                    print("")

                    print(

                        "22 - INCLUIR DOCUMENTO ENCONTRADO!"

                    )



                    print(

                        "Método:",

                        resultado_incluir.get(

                            "metodo"

                        )

                    )



                    print(

                        "Local:",

                        resultado_incluir.get(

                            "caminho"

                        )

                    )



                    print(

                        "23 - INCLUIR DOCUMENTO "

                        "CLICADO COM SUCESSO!"

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando atualização do SEI..."

                )



                print(

                    erro

                )





            time.sleep(

                1

            )





        if not clicou_incluir:



            raise Exception(

                "Não foi possível clicar em "

                "'Incluir Documento'."

            )





        # ============================================================

        # 24. VOLTA PARA DOCUMENTO PRINCIPAL

        # ============================================================



        driver.switch_to.default_content()





        print("")

        print(

            "24 - INCLUIR DOCUMENTO CONCLUÍDO."

        )





        # ============================================================

        # 25. AGUARDA TELA

        # ============================================================



        print("")

        print(

            "25 - Aguardando tela para escolher "

            "o tipo de documento..."

        )





        time.sleep(

            2

        )





        # ============================================================

        # CAMPO EXTERNO

        # ============================================================



        xpath_campo_externo = (

            "/html/body/div[1]/div/div/form/div[2]/input"

        )





        # ============================================================

        # SCRIPT PARA LOCALIZAR XPATH EM QUALQUER FRAME

        # ============================================================



        script_localizar_campo = """

        const xpath = arguments[0];





        function procurar(win, caminho) {



            let doc;



            try {



                doc = win.document;



            } catch (erro) {



                return null;

            }





            try {



                const elemento = doc.evaluate(

                    xpath,

                    doc,

                    null,

                    XPathResult.FIRST_ORDERED_NODE_TYPE,

                    null

                ).singleNodeValue;





                if (elemento) {



                    return caminho;

                }





            } catch (erro) {



            }





            for (

                let i = 0;

                i < win.frames.length;

                i++

            ) {



                try {



                    const resultado = procurar(

                        win.frames[i],

                        caminho.concat([i])

                    );





                    if (resultado !== null) {



                        return resultado;

                    }





                } catch (erro) {



                }

            }





            return null;

        }





        return procurar(

            window,

            []

        );

        """





        print("")

        print(

            "26 - Procurando campo para digitar Externo..."

        )





        caminho_frame_externo = None



        inicio = time.time()





        while time.time() - inicio < TEMPO_MAXIMO_CAMPO_EXTERNO:



            try:



                driver.switch_to.default_content()





                caminho_frame_externo = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_campo_externo

                    )

                )





                if caminho_frame_externo is not None:



                    print(

                        "26 - Campo encontrado!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_externo

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando nova tela...",

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_externo is None:



            raise Exception(

                "Não foi possível localizar o campo "

                "para digitar 'Externo'."

            )





        # ============================================================

        # FUNÇÃO PARA ENTRAR NOS FRAMES

        # ============================================================



        def entrar_no_caminho_frames(caminho):



            driver.switch_to.default_content()





            for indice in caminho:



                frames = driver.find_elements(

                    By.TAG_NAME,

                    "iframe"

                )





                if indice >= len(frames):



                    raise Exception(

                        f"O iframe de índice {indice} "

                        "não existe mais."

                    )





                driver.switch_to.frame(

                    frames[indice]

                )





        # ============================================================

        # 27. ENTRA NO FRAME

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_externo

        )





        # ============================================================

        # 28. CAMPO EXTERNO

        # ============================================================



        campo_externo = WebDriverWait(

            driver,

            20

        ).until(

            EC.element_to_be_clickable(

                (

                    By.XPATH,

                    xpath_campo_externo

                )

            )

        )





        print(

            "27 - Campo pronto para preenchimento."

        )





        # ============================================================

        # 29. LIMPA

        # ============================================================



        campo_externo.click()





        campo_externo.send_keys(

            Keys.CONTROL,

            "a"

        )





        campo_externo.send_keys(

            Keys.BACKSPACE

        )





        # ============================================================

        # 30. DIGITA EXTERNO

        # ============================================================



        print("")

        print(

            "28 - Digitando 'Externo'..."

        )





        for letra in "Externo":



            campo_externo.send_keys(

                letra

            )



            time.sleep(

                0.1

            )





        print(

            "28 - 'Externo' digitado."

        )





        valor_campo = campo_externo.get_attribute(

            "value"

        )





        print(

            "Valor atual do campo:",

            valor_campo

        )





        time.sleep(

            1

        )





        # ============================================================

        # 31. ALT + ENTER

        # ============================================================



        print("")

        print(

            "29 - Pressionando TAB + ENTER..."

        )





        try:



            ActionChains(

                driver

            ).click(

                campo_externo

            ).key_down(

                Keys.TAB

            ).send_keys(

                Keys.ENTER

            ).key_up(

                Keys.TAB

            ).perform()





            print(

                "29 - TAB + ENTER enviado com ActionChains."

            )





        except Exception as erro:



            print(

                "ActionChains falhou:"

            )



            print(

                erro

            )





            entrar_no_caminho_frames(

                caminho_frame_externo

            )





            campo_externo = WebDriverWait(

                driver,

                10

            ).until(

                EC.element_to_be_clickable(

                    (

                        By.XPATH,

                        xpath_campo_externo

                    )

                )

            )





            campo_externo.click()





            campo_externo.send_keys(

                Keys.ALT,

                Keys.ENTER

            )





            print(

                "29 - ALT + ENTER enviado diretamente no campo."

            )





        # ============================================================

        # 32. AGUARDA

        # ============================================================



        time.sleep(

            3

        )





        driver.switch_to.default_content()





        print("")

        print(

            "30 - Externo selecionado. "

            "Procurando Tipo de Documento..."

        )





        # ============================================================

        # 33. NOTA FISCAL

        # ============================================================



        xpath_nota_fiscal = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[4]/select/option[1765]"

        )





        script_nota_fiscal = """

        const xpath = arguments[0];





        function procurar(win, caminho) {



            let doc;



            try {



                doc = win.document;



            } catch (erro) {



                return null;

            }





            try {



                const opcao = doc.evaluate(

                    xpath,

                    doc,

                    null,

                    XPathResult.FIRST_ORDERED_NODE_TYPE,

                    null

                ).singleNodeValue;





                if (opcao) {



                    const select = opcao.parentElement;



                    const texto =

                        (opcao.textContent || "")

                        .trim();





                    opcao.selected = true;





                    if (select) {



                        select.value =

                            opcao.value;





                        select.dispatchEvent(

                            new Event(

                                "input",

                                {

                                    bubbles: true

                                }

                            )

                        );





                        select.dispatchEvent(

                            new Event(

                                "change",

                                {

                                    bubbles: true

                                }

                            )

                        );

                    }





                    return {



                        encontrou: true,



                        selecionou: true,



                        caminho: caminho,



                        texto: texto,



                        valor: opcao.value,



                        indice: opcao.index



                    };

                }





            } catch (erro) {



            }





            for (

                let i = 0;

                i < win.frames.length;

                i++

            ) {



                try {



                    const nomeFrame =

                        win.frames[i].name ||

                        ("frame[" + i + "]");





                    const resultado = procurar(

                        win.frames[i],

                        caminho + " -> " + nomeFrame

                    );





                    if (

                        resultado &&

                        resultado.encontrou

                    ) {



                        return resultado;

                    }





                } catch (erro) {



                }

            }





            return null;

        }





        return procurar(

            window,

            "DOCUMENTO PRINCIPAL"

        );

        """





        print("")

        print(

            "31 - Procurando Nota Fiscal..."

        )





        nota_fiscal_selecionada = False



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                resultado_nota_fiscal = (

                    driver.execute_script(

                        script_nota_fiscal,

                        xpath_nota_fiscal

                    )

                )





                if (

                    resultado_nota_fiscal

                    and

                    resultado_nota_fiscal.get(

                        "encontrou"

                    )

                ):



                    print("")

                    print(

                        "31 - Opção encontrada:"

                    )



                    print(

                        resultado_nota_fiscal.get(

                            "texto"

                        )

                    )



                    print(

                        "Índice:",

                        resultado_nota_fiscal.get(

                            "indice"

                        )

                    )



                    print(

                        "Valor:",

                        resultado_nota_fiscal.get(

                            "valor"

                        )

                    )



                    print(

                        "Local:",

                        resultado_nota_fiscal.get(

                            "caminho"

                        )

                    )





                    if resultado_nota_fiscal.get(

                        "selecionou"

                    ):



                        nota_fiscal_selecionada = True





                        print("")

                        print(

                            "32 - NOTA FISCAL "

                            "SELECIONADA COM SUCESSO!"

                        )





                        break





            except Exception as erro:



                print(

                    "Aguardando campo Tipo de Documento..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if not nota_fiscal_selecionada:



            raise Exception(

                "Não foi possível selecionar "

                "'Nota Fiscal'."

            )





        driver.switch_to.default_content()





        # ============================================================

        # 34. DATA DO DOCUMENTO

        #

        # DF:

        # Data Emissão NF

        #

        # CAMPO:

        # /html/body/div[1]/div/div/form[1]/div[4]/input

        # ============================================================



        print("")

        print(

            "33 - Preparando Data do Documento..."

        )





        # ============================================================

        # PEGA DATA DA MESMA LINHA DO PROCESSO

        # ============================================================



        valor_data_emissao = df.loc[

            indice_linha_atual,

            "Data Emissão NF"

        ]





        if pd.isna(

            valor_data_emissao

        ):



            raise Exception(

                "A coluna 'Data Emissão NF' "

                f"está vazia na linha {indice_linha_atual}."

            )





        # ============================================================

        # FORMATA DATA

        # ============================================================



        if isinstance(

            valor_data_emissao,

            pd.Timestamp

        ):



            data_documento = (

                valor_data_emissao.strftime(

                    "%d/%m/%Y"

                )

            )





        else:



            texto_data = str(

                valor_data_emissao

            ).strip()





            try:



                data_convertida = pd.to_datetime(

                    texto_data,

                    dayfirst=True,

                    errors="raise"

                )





                data_documento = (

                    data_convertida.strftime(

                        "%d/%m/%Y"

                    )

                )





            except Exception:



                data_documento = texto_data





        print("")

        print(

            "Data Emissão NF original:",

            valor_data_emissao

        )



        print(

            "Data formatada:",

            data_documento

        )





        # ============================================================

        # XPATH DATA

        # ============================================================



        xpath_data_documento = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[4]/input"

        )





        # ============================================================

        # 35. LOCALIZA CAMPO DATA

        # ============================================================



        print("")

        print(

            "34 - Procurando campo Data do Documento..."

        )





        caminho_frame_data = None



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                caminho_frame_data = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_data_documento

                    )

                )





                if caminho_frame_data is not None:



                    print(

                        "34 - Campo Data do Documento encontrado!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_data

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando campo Data do Documento..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_data is None:



            raise Exception(

                "Não foi possível localizar "

                "o campo Data do Documento."

            )





        # ============================================================

        # 36. ENTRA NO FRAME

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_data

        )





        # ============================================================

        # 37. LOCALIZA INPUT DATA

        # ============================================================



        campo_data_documento = WebDriverWait(

            driver,

            20

        ).until(

            EC.element_to_be_clickable(

                (

                    By.XPATH,

                    xpath_data_documento

                )

            )

        )





        print(

            "35 - Campo Data do Documento pronto."

        )





        # ============================================================

        # 38. LIMPA DATA

        # ============================================================



        campo_data_documento.click()





        campo_data_documento.send_keys(

            Keys.CONTROL,

            "a"

        )





        campo_data_documento.send_keys(

            Keys.BACKSPACE

        )





        time.sleep(

            0.3

        )





        # ============================================================

        # 39. DIGITA DATA

        # ============================================================



        campo_data_documento.send_keys(

            data_documento

        )





        print("")

        print(

            "36 - DATA DO DOCUMENTO PREENCHIDA!"

        )



        print(

            "Valor inserido:",

            data_documento

        )





        # ============================================================

        # 40. CONFERE DATA

        # ============================================================



        valor_final_data = (

            campo_data_documento.get_attribute(

                "value"

            )

        )





        print(

            "Valor atual do campo:",

            valor_final_data

        )





        # ============================================================

        # 41. TAB

        # ============================================================



        campo_data_documento.send_keys(

            Keys.TAB

        )





        time.sleep(

            1

        )





        driver.switch_to.default_content()





        # ============================================================

        # 42. CAMPO NFS

        #

        # NOVA ETAPA

        #

        # DF:

        # NFS

        #

        # CAMPO:

        # /html/body/div[1]/div/div/form[1]/div[7]/input[1]

        # ============================================================



        print("")

        print(

            "37 - Preparando preenchimento do campo NFS..."

        )





        # ============================================================

        # PEGA NFS DA MESMA LINHA DO PROCESSO

        # ============================================================



        valor_nfs = df.loc[

            indice_linha_atual,

            "NFS"

        ]





        if pd.isna(

            valor_nfs

        ):



            raise Exception(

                "A coluna 'NFS' está vazia "

                f"na linha {indice_linha_atual}."

            )





        # ============================================================

        # CONVERTE NFS PARA TEXTO

        #

        # EXEMPLO:

        # 12345.0 -> 12345

        # ============================================================



        if (

            isinstance(

                valor_nfs,

                float

            )

            and

            valor_nfs.is_integer()

        ):



            nfs = str(

                int(

                    valor_nfs

                )

            )



        else:



            nfs = str(

                valor_nfs

            ).strip()





        print(

            "NFS obtido do DataFrame:",

            nfs

        )





        # ============================================================

        # XPATH NFS

        # ============================================================



        xpath_campo_nfs = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[7]/input[1]"

        )





        # ============================================================

        # 43. LOCALIZA O CAMPO NFS

        # ============================================================



        print("")

        print(

            "38 - Procurando campo NFS..."

        )





        caminho_frame_nfs = None



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                caminho_frame_nfs = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_campo_nfs

                    )

                )





                if caminho_frame_nfs is not None:



                    print(

                        "38 - Campo NFS encontrado!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_nfs

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando campo NFS..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_nfs is None:



            raise Exception(

                "Não foi possível localizar "

                "o campo NFS."

            )





        # ============================================================

        # 44. ENTRA NO FRAME DO NFS

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_nfs

        )





        # ============================================================

        # 45. LOCALIZA INPUT NFS

        # ============================================================



        campo_nfs = WebDriverWait(

            driver,

            20

        ).until(

            EC.element_to_be_clickable(

                (

                    By.XPATH,

                    xpath_campo_nfs

                )

            )

        )





        print(

            "39 - Campo NFS pronto para preenchimento."

        )





        # ============================================================

        # 46. LIMPA CAMPO NFS

        # ============================================================



        campo_nfs.click()





        campo_nfs.send_keys(

            Keys.CONTROL,

            "a"

        )





        campo_nfs.send_keys(

            Keys.BACKSPACE

        )





        time.sleep(

            0.3

        )





        # ============================================================

        # 47. DIGITA NFS

        # ============================================================



        campo_nfs.send_keys(

            nfs

        )





        print("")

        print(

            "40 - NFS PREENCHIDO!"

        )



        print(

            "Valor inserido:",

            nfs

        )





        # ============================================================

        # 48. CONFERE VALOR NFS

        # ============================================================



        valor_final_nfs = (

            campo_nfs.get_attribute(

                "value"

            )

        )





        print(

            "Valor atual do campo NFS:",

            valor_final_nfs

        )





        # ============================================================

        # 49. TAB PARA CONFIRMAR

        # ============================================================



        campo_nfs.send_keys(

            Keys.TAB

        )





        time.sleep(

            1

        )





        # ============================================================

        # 50. CAMPO NOME NA ÁRVORE

        #

        # XPATH:

        # /html/body/div[1]/div/div/form[1]/div[7]/input[2]

        #

        # VALOR:

        # df["CONTRATO"]

        # ============================================================



        print("")

        print(

            "41 - Preparando preenchimento do campo Nome na Árvore..."

        )





        # ============================================================

        # VALIDA COLUNA CONTRATO

        # ============================================================



        if "CONTRATO" not in df.columns:



            raise Exception(

                "A coluna 'CONTRATO' não foi encontrada no DataFrame."

            )





        # ============================================================

        # PEGA CONTRATO DA MESMA LINHA DO PROCESSO SEI

        # ============================================================



        valor_contrato = df.loc[

            indice_linha_atual,

            "CONTRATO"

        ]





        # ============================================================

        # VALIDA VALOR

        # ============================================================



        if pd.isna(

            valor_contrato

        ):



            raise Exception(

                "A coluna 'CONTRATO' está vazia "

                f"na linha {indice_linha_atual}."

            )





        # ============================================================

        # CONVERTE CONTRATO PARA TEXTO

        #

        # Exemplo:

        # 12345.0 -> 12345

        #

        # Se já estiver como texto:

        # 001/2026 -> mantém 001/2026

        # ============================================================



        if (

            isinstance(

                valor_contrato,

                float

            )

            and

            valor_contrato.is_integer()

        ):



            contrato = str(

                int(

                    valor_contrato

                )

            )



        else:



            contrato = str(

                valor_contrato

            ).strip()





        print(

            "CONTRATO obtido do DataFrame:",

            contrato

        )





        # ============================================================

        # XPATH NOME NA ÁRVORE

        # ============================================================



        xpath_nome_arvore = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[7]/input[2]"

        )





        # ============================================================

        # 51. LOCALIZA O CAMPO NOME NA ÁRVORE

        #

        # REUTILIZA:

        # script_localizar_campo

        # ============================================================



        print("")

        print(

            "42 - Procurando campo Nome na Árvore..."

        )





        caminho_frame_nome_arvore = None



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                caminho_frame_nome_arvore = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_nome_arvore

                    )

                )





                if caminho_frame_nome_arvore is not None:



                    print(

                        "42 - Campo Nome na Árvore encontrado!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_nome_arvore

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando campo Nome na Árvore..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_nome_arvore is None:



            raise Exception(

                "Não foi possível localizar "

                "o campo Nome na Árvore."

            )





        # ============================================================

        # 52. ENTRA NO FRAME CORRETO

        #

        # REUTILIZA:

        # entrar_no_caminho_frames()

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_nome_arvore

        )





        # ============================================================

        # 53. LOCALIZA INPUT NOME NA ÁRVORE

        # ============================================================



        campo_nome_arvore = WebDriverWait(

            driver,

            20

        ).until(

            EC.element_to_be_clickable(

                (

                    By.XPATH,

                    xpath_nome_arvore

                )

            )

        )





        print(

            "43 - Campo Nome na Árvore pronto para preenchimento."

        )





        # ============================================================

        # 54. LIMPA O CAMPO

        # ============================================================



        campo_nome_arvore.click()





        campo_nome_arvore.send_keys(

            Keys.CONTROL,

            "a"

        )





        campo_nome_arvore.send_keys(

            Keys.BACKSPACE

        )





        time.sleep(

            0.3

        )





        # ============================================================

        # 55. DIGITA CONTRATO

        # ============================================================



        campo_nome_arvore.send_keys(

            contrato

        )





        print("")

        print(

            "44 - NOME NA ÁRVORE PREENCHIDO!"

        )



        print(

            "Valor inserido:",

            contrato

        )





        # ============================================================

        # 56. CONFERE VALOR

        # ============================================================



        valor_final_nome_arvore = (

            campo_nome_arvore.get_attribute(

                "value"

            )

        )





        print(

            "Valor atual do campo Nome na Árvore:",

            valor_final_nome_arvore

        )





        # ============================================================

        # 57. TAB PARA CONFIRMAR

        # ============================================================



        campo_nome_arvore.send_keys(

            Keys.TAB

        )





        time.sleep(

            1

        )



        # ============================================================

        # 58. CAMPO FORMATO = NATO-DIGITAL

        #

        # XPATH:

        # /html/body/div[1]/div/div/form[1]/div[8]/

        # fieldset/div[1]/div/label

        # ============================================================



        print("")

        print(

            "45 - Selecionando Formato: Nato-digital..."

        )





        # ============================================================

        # XPATH NATO-DIGITAL

        # ============================================================



        xpath_nato_digital = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[8]/fieldset/"

            "div[1]/div/label"

        )





        # ============================================================

        # 59. LOCALIZA EM QUAL FRAME ESTÁ A OPÇÃO

        #

        # REUTILIZA:

        # script_localizar_campo

        # ============================================================



        caminho_frame_nato_digital = None



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                caminho_frame_nato_digital = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_nato_digital

                    )

                )





                if caminho_frame_nato_digital is not None:



                    print(

                        "45 - Opção Nato-digital encontrada!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_nato_digital

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando campo Formato..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_nato_digital is None:



            raise Exception(

                "Não foi possível localizar "

                "a opção Nato-digital."

            )





        # ============================================================

        # 60. ENTRA NO FRAME CORRETO

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_nato_digital

        )





        # ============================================================

        # 61. LOCALIZA O LABEL NATO-DIGITAL

        # ============================================================



        label_nato_digital = WebDriverWait(

            driver,

            20

        ).until(

            EC.presence_of_element_located(

                (

                    By.XPATH,

                    xpath_nato_digital

                )

            )

        )





        print(

            "46 - Campo Formato pronto."

        )





        # ============================================================

        # MOSTRA O TEXTO ENCONTRADO

        # ============================================================



        try:



            texto_formato = (

                label_nato_digital.text

                .strip()

            )



        except Exception:



            texto_formato = ""





        print(

            "Opção encontrada:",

            texto_formato

        )





        # ============================================================

        # 62. ROLA ATÉ A OPÇÃO

        # ============================================================



        driver.execute_script(

            """

            arguments[0].scrollIntoView({

                block: 'center',

                inline: 'center'

            });

            """,

            label_nato_digital

        )





        time.sleep(

            0.5

        )





        # ============================================================

        # 63. CLICA NATO-DIGITAL

        # ============================================================



        try:



            label_nato_digital.click()





            print(

                "47 - Nato-digital clicado."

            )





        except Exception:



            print(

                "Clique normal não funcionou. "

                "Tentando JavaScript..."

            )





            driver.execute_script(

                "arguments[0].click();",

                label_nato_digital

            )





            print(

                "47 - Nato-digital clicado via JavaScript."

            )





        # ============================================================

        # 64. CONFERE O RADIO ASSOCIADO AO LABEL

        # ============================================================



        time.sleep(

            0.5

        )





        resultado_formato = driver.execute_script(

            """

            const label = arguments[0];



            let input = null;



            // Se o label tiver atributo for

            if (label.htmlFor) {



                input = document.getElementById(

                    label.htmlFor

                );

            }



            // Caso o input esteja dentro do label

            if (!input) {



                input = label.querySelector(

                    "input[type='radio']"

                );

            }



            // Caso esteja próximo ao label

            if (!input) {



                const container = label.parentElement;



                if (container) {



                    input = container.querySelector(

                        "input[type='radio']"

                    );

                }

            }





            return {



                texto:

                    (label.textContent || "").trim(),



                encontrouRadio:

                    !!input,



                marcado:

                    input ? input.checked : null,



                valor:

                    input ? input.value : null

            };

            """,

            label_nato_digital

        )





        print(

            "Resultado da seleção:",

            resultado_formato

        )





        # ============================================================

        # 65. VALIDA

        # ============================================================



        if (

            resultado_formato

            and

            resultado_formato.get(

                "encontrouRadio"

            )

            and

            resultado_formato.get(

                "marcado"

            ) is False

        ):



            raise Exception(

                "A opção Nato-digital foi localizada, "

                "mas o radio não ficou selecionado."

            )







        # ============================================================

        # 66. NÍVEL DE ACESSO = PÚBLICO

        #

        # XPATH:

        # /html/body/div[1]/div/div/form[1]/div[14]/

        # fieldset/div[3]/div/label

        # ============================================================



        print("")

        print(

            "48 - Selecionando Nível de Acesso: Público..."

        )





        # ============================================================

        # XPATH PÚBLICO

        # ============================================================



        xpath_nivel_publico = (

            "/html/body/div[1]/div/div/"

            "form[1]/div[14]/fieldset/"

            "div[3]/div/label"

        )





        # ============================================================

        # 67. LOCALIZA EM QUAL FRAME ESTÁ O CAMPO

        #

        # REUTILIZA:

        # script_localizar_campo

        # ============================================================



        caminho_frame_publico = None



        inicio = time.time()





        while time.time() - inicio < 30:



            try:



                driver.switch_to.default_content()





                caminho_frame_publico = (

                    driver.execute_script(

                        script_localizar_campo,

                        xpath_nivel_publico

                    )

                )





                if caminho_frame_publico is not None:



                    print(

                        "48 - Opção Público encontrada!"

                    )



                    print(

                        "Caminho dos frames:",

                        caminho_frame_publico

                    )



                    break





            except Exception as erro:



                print(

                    "Aguardando campo Nível de Acesso..."

                )



                print(

                    erro

                )





            time.sleep(

                0.5

            )





        if caminho_frame_publico is None:



            raise Exception(

                "Não foi possível localizar "

                "a opção Público."

            )





        # ============================================================

        # 68. ENTRA NO FRAME CORRETO

        # ============================================================



        entrar_no_caminho_frames(

            caminho_frame_publico

        )





        # ============================================================

        # 69. LOCALIZA O LABEL PÚBLICO

        # ============================================================



        label_publico = WebDriverWait(

            driver,

            20

        ).until(

            EC.presence_of_element_located(

                (

                    By.XPATH,

                    xpath_nivel_publico

                )

            )

        )





        print(

            "49 - Campo Nível de Acesso pronto."

        )





        # ============================================================

        # MOSTRA O TEXTO ENCONTRADO

        # ============================================================



        try:



            texto_nivel = (

                label_publico.text

                .strip()

            )



        except Exception:



            texto_nivel = ""





        print(

            "Opção encontrada:",

            texto_nivel

        )





        # ============================================================

        # 70. ROLA ATÉ A OPÇÃO

        # ============================================================



        driver.execute_script(

            """

            arguments[0].scrollIntoView({

                block: 'center',

                inline: 'center'

            });

            """,

            label_publico

        )





        time.sleep(

            0.5

        )





        # ============================================================

        # 71. CLICA EM PÚBLICO

        # ============================================================



        try:



            label_publico.click()





            print(

                "50 - Público clicado."

            )





        except Exception:



            print(

                "Clique normal não funcionou. "

                "Tentando JavaScript..."

            )





            driver.execute_script(

                "arguments[0].click();",

                label_publico

            )





            print(

                "50 - Público clicado via JavaScript."

            )





        # ============================================================

        # 72. CONFERE O RADIO ASSOCIADO

        # ============================================================



        time.sleep(

            0.5

        )





        resultado_publico = driver.execute_script(

            """

            const label = arguments[0];



            let input = null;





            // Se label possuir atributo for

            if (label.htmlFor) {



                input = document.getElementById(

                    label.htmlFor

                );

            }





            // Input dentro do próprio label

            if (!input) {



                input = label.querySelector(

                    "input[type='radio']"

                );

            }





            // Input próximo ao label

            if (!input) {



                const container =

                    label.parentElement;



                if (container) {



                    input = container.querySelector(

                        "input[type='radio']"

                    );

                }

            }





            return {



                texto:

                    (label.textContent || "").trim(),



                encontrouRadio:

                    !!input,



                marcado:

                    input ? input.checked : null,



                valor:

                    input ? input.value : null

            };

            """,

            label_publico

        )





        print(

            "Resultado da seleção:",

            resultado_publico

        )





        # ============================================================

        # 73. VALIDA

        # ============================================================



        if (

            resultado_publico

            and

            resultado_publico.get(

                "encontrouRadio"

            )

            and

            resultado_publico.get(

                "marcado"

            ) is False

        ):



            raise Exception(

                "A opção Público foi localizada, "

                "mas o radio não ficou selecionado."

            )





        # ============================================================


        # 74. ANEXAR ARQUIVO PDF AUTOMATICAMENTE
        #
        # DIRETÓRIO:
        # X:\GOB\Teste Automação - NÃO DELETAR
        #
        # DF:
        # coluna "Nota Fiscal Eletrônica"
        #
        # O nome no DataFrame vem sem .pdf.
        # ============================================================

        print("")
        print(
            "54 - Preparando anexo da Nota Fiscal Eletrônica..."
        )


        # ============================================================
        # 75. VALIDA A COLUNA
        # ============================================================

        if "Nota Fiscal Eletrônica" not in df.columns:

            raise Exception(
                "A coluna 'Nota Fiscal Eletrônica' "
                "não foi encontrada no DataFrame."
            )


        # ============================================================
        # 76. PEGA O NOME DO ARQUIVO DA MESMA LINHA DO PROCESSO
        # ============================================================

        valor_nota_fiscal_eletronica = df.loc[
            indice_linha_atual,
            "Nota Fiscal Eletrônica"
        ]


        if pd.isna(
            valor_nota_fiscal_eletronica
        ):

            raise Exception(
                "A coluna 'Nota Fiscal Eletrônica' está vazia "
                f"na linha {indice_linha_atual}."
            )


        # ============================================================
        # 77. CONVERTE O NOME PARA TEXTO
        # ============================================================

        if (
            isinstance(
                valor_nota_fiscal_eletronica,
                float
            )
            and
            valor_nota_fiscal_eletronica.is_integer()
        ):

            nome_nota_fiscal = str(
                int(
                    valor_nota_fiscal_eletronica
                )
            )

        else:

            nome_nota_fiscal = str(
                valor_nota_fiscal_eletronica
            ).strip()


        # ============================================================
        # 78. CONCATENA .pdf
        # ============================================================

        if nome_nota_fiscal.lower().endswith(
            ".pdf"
        ):

            nome_arquivo_pdf = nome_nota_fiscal

        else:

            nome_arquivo_pdf = (
                nome_nota_fiscal
                + ".pdf"
            )


        print(
            "Nome da Nota Fiscal no DF:",
            nome_nota_fiscal
        )

        print(
            "Nome do arquivo PDF:",
            nome_arquivo_pdf
        )


        # ============================================================
        # 79. DIRETÓRIO DOS ARQUIVOS
        # ============================================================

        diretorio_notas = (
            r"X:\GOB\Teste Automação - NÃO DELETAR"
        )


        if not os.path.isdir(
            diretorio_notas
        ):

            raise Exception(
                "O diretório das Notas Fiscais não foi encontrado:\n"
                f"{diretorio_notas}"
            )


        # ============================================================
        # 80. PROCURA O PDF PRIMEIRO NA PASTA PRINCIPAL
        # ============================================================

        caminho_arquivo_pdf = os.path.join(
            diretorio_notas,
            nome_arquivo_pdf
        )


        if not os.path.isfile(
            caminho_arquivo_pdf
        ):

            print("")
            print(
                "55 - PDF não encontrado diretamente na pasta principal."
            )

            print(
                "Procurando também nas subpastas..."
            )


            arquivos_encontrados = []


            for raiz, pastas, arquivos in os.walk(
                diretorio_notas
            ):

                for arquivo in arquivos:

                    if (
                        arquivo.lower()
                        ==
                        nome_arquivo_pdf.lower()
                    ):

                        arquivos_encontrados.append(
                            os.path.join(
                                raiz,
                                arquivo
                            )
                        )


            if len(
                arquivos_encontrados
            ) == 0:

                raise Exception(
                    "O PDF da Nota Fiscal não foi encontrado.\n\n"
                    f"Arquivo procurado: {nome_arquivo_pdf}\n"
                    f"Diretório: {diretorio_notas}"
                )


            if len(
                arquivos_encontrados
            ) > 1:

                print("")
                print(
                    "Foram encontrados vários PDFs com o mesmo nome:"
                )


                for arquivo_encontrado in arquivos_encontrados:

                    print(
                        arquivo_encontrado
                    )


                raise Exception(
                    "Foi encontrado mais de um arquivo PDF com o nome "
                    f"'{nome_arquivo_pdf}'."
                )


            caminho_arquivo_pdf = (
                arquivos_encontrados[0]
            )


        caminho_arquivo_pdf = os.path.abspath(
            caminho_arquivo_pdf
        )


        print("")
        print(
            "55 - PDF localizado com sucesso!"
        )

        print(
            "Caminho completo:",
            caminho_arquivo_pdf
        )


        # ============================================================
        # 81. XPATH DO CAMPO ANEXAR ARQUIVO
        # ============================================================

        xpath_anexar_arquivo = (
            "/html/body/div[1]/div/div/"
            "form[2]/div[1]/label"
        )


        # ============================================================
        # 82. LOCALIZA EM QUAL FRAME ESTÁ O CAMPO ANEXAR ARQUIVO
        #
        # Reutiliza:
        # script_localizar_campo
        # ============================================================

        print("")
        print(
            "56 - Procurando campo Anexar Arquivo..."
        )


        caminho_frame_anexar = None

        inicio = time.time()


        while time.time() - inicio < 30:

            try:

                driver.switch_to.default_content()


                caminho_frame_anexar = (
                    driver.execute_script(
                        script_localizar_campo,
                        xpath_anexar_arquivo
                    )
                )


                if caminho_frame_anexar is not None:

                    print(
                        "56 - Campo Anexar Arquivo encontrado!"
                    )

                    print(
                        "Caminho dos frames:",
                        caminho_frame_anexar
                    )

                    break


            except Exception as erro:

                print(
                    "Aguardando campo Anexar Arquivo..."
                )

                print(
                    erro
                )


            time.sleep(
                0.5
            )


        if caminho_frame_anexar is None:

            raise Exception(
                "Não foi possível localizar "
                "o campo Anexar Arquivo."
            )


        # ============================================================
        # 83. ENTRA NO FRAME CORRETO
        #
        # Reutiliza:
        # entrar_no_caminho_frames()
        # ============================================================

        entrar_no_caminho_frames(
            caminho_frame_anexar
        )


        # ============================================================
        # 84. LOCALIZA O LABEL ANEXAR ARQUIVO
        # ============================================================

        label_anexar_arquivo = WebDriverWait(
            driver,
            20
        ).until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    xpath_anexar_arquivo
                )
            )
        )


        print(
            "57 - Campo Anexar Arquivo pronto."
        )


        # ============================================================
        # 85. LOCALIZA O INPUT TYPE="FILE"
        #
        # IMPORTANTE:
        # NÃO CLICA NO LABEL PARA NÃO ABRIR A JANELA NATIVA
        # DO WINDOWS. O ARQUIVO É ENVIADO DIRETAMENTE AO INPUT.
        # ============================================================

        input_arquivo = None


        # Primeiro tenta usar o atributo "for" do label.
        try:

            id_input_arquivo = (
                label_anexar_arquivo.get_attribute(
                    "for"
                )
            )


            if id_input_arquivo:

                elementos_input = driver.find_elements(
                    By.ID,
                    id_input_arquivo
                )


                if elementos_input:

                    input_arquivo = (
                        elementos_input[0]
                    )


        except Exception:

            pass


        # Depois procura no formulário ao qual o label pertence.
        if input_arquivo is None:

            try:

                formulario_upload = (
                    label_anexar_arquivo.find_element(
                        By.XPATH,
                        "./ancestor::form[1]"
                    )
                )


                elementos_input = (
                    formulario_upload.find_elements(
                        By.CSS_SELECTOR,
                        "input[type='file']"
                    )
                )


                if elementos_input:

                    input_arquivo = (
                        elementos_input[0]
                    )


            except Exception:

                pass


        # Fallback: procura qualquer input file no frame atual.
        if input_arquivo is None:

            elementos_input = driver.find_elements(
                By.CSS_SELECTOR,
                "input[type='file']"
            )


            if elementos_input:

                input_arquivo = (
                    elementos_input[0]
                )


        if input_arquivo is None:

            raise Exception(
                "O campo real <input type='file'> "
                "não foi encontrado no formulário de anexo."
            )


        print("")
        print(
            "58 - Input real de upload encontrado."
        )


        # ============================================================
        # 86. ENVIA O PDF DIRETAMENTE PARA O INPUT FILE
        # ============================================================

        print(
            "59 - Anexando PDF:"
        )

        print(
            caminho_arquivo_pdf
        )


        try:

            input_arquivo.send_keys(
                caminho_arquivo_pdf
            )


        except Exception as erro_upload:

            print(
                "Primeira tentativa de upload falhou."
            )

            print(
                erro_upload
            )

            print(
                "Tentando tornar o input acessível..."
            )


            driver.execute_script(
                """
                const input = arguments[0];

                input.style.display = 'block';
                input.style.visibility = 'visible';
                input.style.opacity = '1';
                """,
                input_arquivo
            )


            input_arquivo.send_keys(
                caminho_arquivo_pdf
            )


        print("")
        print(
            "60 - PDF ENVIADO PARA O CAMPO ANEXAR ARQUIVO!"
        )


        # ============================================================
        # 87. AGUARDA O SEI PROCESSAR O ARQUIVO
        # ============================================================

        time.sleep(
            3
        )


        # ============================================================
        # 88. CONFERE O VALOR DO INPUT
        # ============================================================

        try:

            valor_input_arquivo = (
                input_arquivo.get_attribute(
                    "value"
                )
            )


            print(
                "Valor atual do campo de arquivo:",
                valor_input_arquivo
            )


        except StaleElementReferenceException:

            print(
                "O SEI atualizou o campo após receber o arquivo."
            )


        except Exception as erro:

            print(
                "Não foi possível consultar o valor final do campo de arquivo."
            )

            print(
                erro
            )


        # ============================================================
        # 89. VOLTA PARA O DOCUMENTO PRINCIPAL
        # ============================================================

        driver.switch_to.default_content()


        # ============================================================
        # FINAL
        # ============================================================

        print("")
        print(
            "=============================================="
        )

        print(
            "FLUXO EXECUTADO COM SUCESSO"
        )

        print(
            "=============================================="
        )

        print(
            "Processo SEI:",
            processo_sei
        )

        print(
            "Origem: Externo"
        )

        print(
            "Tipo de Documento: Nota Fiscal"
        )

        print(
            "Data do Documento:",
            data_documento
        )

        print(
            "NFS:",
            nfs
        )

        print(
            "CONTRATO:",
            contrato
        )

        print(
            "Formato: Nato-digital"
        )

        print(
            "Nível de Acesso: Público"
        )

        print(
            "Nota Fiscal Eletrônica:",
            nome_nota_fiscal
        )

        print(
            "PDF anexado:",
            caminho_arquivo_pdf
        )

        print(
            "=============================================="
        )

        # ============================================================
        # 90. SALVAR DOCUMENTO APÓS ANEXAR O PDF
        #
        # XPATH:
        # /html/body/div[1]/div/div/div[2]/button[1]
        # ============================================================

        print("")
        print(
            "61 - Procurando botão Salvar..."
        )

        xpath_salvar_final = (
            "/html/body/div[1]/div/div/"
            "div[2]/button[1]"
        )

        caminho_frame_salvar_final = None
        inicio = time.time()

        while time.time() - inicio < 30:

            try:

                driver.switch_to.default_content()

                caminho_frame_salvar_final = (
                    driver.execute_script(
                        script_localizar_campo,
                        xpath_salvar_final
                    )
                )

                if caminho_frame_salvar_final is not None:

                    print(
                        "61 - Botão Salvar encontrado!"
                    )

                    print(
                        "Caminho dos frames:",
                        caminho_frame_salvar_final
                    )

                    break

            except Exception as erro:

                print(
                    "Aguardando botão Salvar..."
                )

                print(
                    erro
                )

            time.sleep(
                0.5
            )


        if caminho_frame_salvar_final is None:

            raise Exception(
                "Não foi possível localizar "
                "o botão Salvar."
            )


        entrar_no_caminho_frames(
            caminho_frame_salvar_final
        )


        botao_salvar_final = WebDriverWait(
            driver,
            20
        ).until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    xpath_salvar_final
                )
            )
        )


        print(
            "62 - Botão Salvar pronto para clique."
        )


        driver.execute_script(
            """
            arguments[0].scrollIntoView({
                block: 'center',
                inline: 'center'
            });
            """,
            botao_salvar_final
        )

        time.sleep(
            0.5
        )


        try:

            botao_salvar_final.click()

            print("")
            print(
                "63 - BOTÃO SALVAR CLICADO!"
            )

        except Exception as erro:

            print(
                "Clique normal não funcionou."
            )

            print(
                "Tentando clique via JavaScript..."
            )

            print(
                erro
            )

            driver.execute_script(
                "arguments[0].click();",
                botao_salvar_final
            )

            print("")
            print(
                "63 - BOTÃO SALVAR CLICADO VIA JAVASCRIPT!"
            )


        # ============================================================
        # 91. TRATA CONFIRMAÇÃO DO SEI APÓS CLICAR EM SALVAR
        #
        # Se o SEI exibir um alerta de documento já cadastrado com a
        # pergunta "Deseja continuar?", confirma com alert.accept().
        # Isso equivale a selecionar OK/pressionar ENTER no alerta.
        # Outras mensagens de alerta não são confirmadas automaticamente.
        # ============================================================

        def confirmar_alerta_duplicidade(timeout=4):
            try:
                alerta = WebDriverWait(driver, timeout).until(EC.alert_is_present())
            except TimeoutException:
                return False

            texto_alerta = alerta.text.strip()
            print("Alerta exibido pelo SEI:", texto_alerta)

            if (
                "já existe um documento" in texto_alerta.casefold()
                and "deseja continuar" in texto_alerta.casefold()
            ):
                alerta.accept()  # Confirma o OK (equivalente a ENTER)
                print("64 - Alerta de duplicidade confirmado (OK/ENTER).")
                return True

            raise RuntimeError(
                "O SEI apresentou um alerta diferente do esperado; "
                "não foi confirmado automaticamente: " + texto_alerta
            )


        print("")
        print("64 - Verificando confirmação após clicar em Salvar...")

        alerta_confirmado = confirmar_alerta_duplicidade(timeout=5)

        if not alerta_confirmado:
            # Mantém a solicitação original de pressionar ENTER após Salvar
            # apenas quando não há um alerta aberto naquele momento.
            try:
                driver.switch_to.active_element.send_keys(Keys.ENTER)
                print("64 - ENTER enviado ao elemento ativo.")
            except (WebDriverException, StaleElementReferenceException) as erro:
                # Um alerta pode surgir entre a verificação e o envio da tecla.
                if not confirmar_alerta_duplicidade(timeout=2):
                    print("Não foi possível enviar ENTER:", erro)
                    raise

            # Confere se o ENTER produziu uma confirmação posterior.
            confirmar_alerta_duplicidade(timeout=3)

        print("")
        print("64 - Aguardando o SEI processar o salvamento...")
        time.sleep(5)

        # Antes de mudar de frame, trata eventual confirmação atrasada.
        confirmar_alerta_duplicidade(timeout=2)
        driver.switch_to.default_content()

        print("")
        print("==============================================")
        print("SOLICITAÇÃO DE SALVAMENTO ENVIADA AO SEI")
        print("==============================================")
        print("Processo SEI:", processo_sei)
        print("PDF:", nome_arquivo_pdf)
        print("==============================================")

        # ============================================================
        # 97. CLICAR NOVAMENTE EM INCLUIR DOCUMENTO
        #
        # XPATH:
        # /html/body/div[1]/div/div/div[2]/div/a[1]/img
        # ============================================================

        print("")
        print(
            "65 - Procurando novamente Incluir Documento..."
        )


        # ============================================================
        # XPATH INCLUIR DOCUMENTO
        # ============================================================

        xpath_incluir_documento_novo = (
            "/html/body/div[1]/div/div/"
            "div[2]/div/a[1]/img"
        )


        # ============================================================
        # 98. LOCALIZA EM QUAL FRAME ESTÁ O BOTÃO
        #
        # REUTILIZA:
        # script_localizar_campo
        # ============================================================

        caminho_frame_incluir_novo = None

        inicio = time.time()


        while time.time() - inicio < 30:

            try:

                driver.switch_to.default_content()


                caminho_frame_incluir_novo = (
                    driver.execute_script(
                        script_localizar_campo,
                        xpath_incluir_documento_novo
                    )
                )


                if caminho_frame_incluir_novo is not None:

                    print(
                        "65 - Incluir Documento encontrado!"
                    )

                    print(
                        "Caminho dos frames:",
                        caminho_frame_incluir_novo
                    )

                    break


            except Exception as erro:

                print(
                    "Aguardando botão Incluir Documento..."
                )

                print(
                    erro
                )


            time.sleep(
                0.5
            )


        if caminho_frame_incluir_novo is None:

            raise Exception(
                "Não foi possível localizar "
                "o botão Incluir Documento."
            )


        # ============================================================
        # 99. ENTRA NO FRAME CORRETO
        #
        # REUTILIZA:
        # entrar_no_caminho_frames()
        # ============================================================

        entrar_no_caminho_frames(
            caminho_frame_incluir_novo
        )


        # ============================================================
        # 100. LOCALIZA O BOTÃO INCLUIR DOCUMENTO
        # ============================================================

        botao_incluir_documento_novo = WebDriverWait(
            driver,
            20
        ).until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    xpath_incluir_documento_novo
                )
            )
        )


        print(
            "66 - Botão Incluir Documento pronto para clique."
        )


        # ============================================================
        # 101. ROLA ATÉ O BOTÃO
        # ============================================================

        driver.execute_script(
            """
            arguments[0].scrollIntoView({
                block: 'center',
                inline: 'center'
            });
            """,
            botao_incluir_documento_novo
        )


        time.sleep(
            0.5
        )


        # ============================================================
        # 102. CLICA EM INCLUIR DOCUMENTO
        # ============================================================

        try:

            botao_incluir_documento_novo.click()


            print("")
            print(
                "67 - INCLUIR DOCUMENTO CLICADO!"
            )


        except Exception as erro:

            print(
                "Clique normal não funcionou."
            )

            print(
                "Tentando clique via JavaScript..."
            )

            print(
                erro
            )


            driver.execute_script(
                "arguments[0].click();",
                botao_incluir_documento_novo
            )


            print("")
            print(
                "67 - INCLUIR DOCUMENTO CLICADO VIA JAVASCRIPT!"
            )


        # ============================================================
        # 103. AGUARDA NOVA TELA
        # ============================================================

        print("")
        print(
            "68 - Aguardando abertura da tela "
            "de inclusão de documento..."
        )


        time.sleep(
            3
        )


        # ============================================================
        # VOLTA PARA DOCUMENTO PRINCIPAL
        # ============================================================

        driver.switch_to.default_content()


        print("")
        print(
            "=============================================="
        )

        print(
            "INCLUIR DOCUMENTO CLICADO COM SUCESSO"
        )

        print(
            "=============================================="
        )

        # ============================================================
        # 104. NOVO DOCUMENTO = AUTORIZAÇÃO
        #
        # CAMPO:
        # /html/body/div[1]/div/div/form/div[2]/input
        #
        # AÇÃO:
        # - Clicar no campo
        # - Digitar "Autorização"
        # - Pressionar TAB + ENTER
        # ============================================================

        print("")
        print(
            "69 - Procurando campo para digitar Autorização..."
        )


        # ============================================================
        # XPATH DO CAMPO
        # ============================================================

        xpath_campo_autorizacao = (
            "/html/body/div[1]/div/div/form/div[2]/input"
        )


        # ============================================================
        # 105. LOCALIZA EM QUAL FRAME ESTÁ O CAMPO
        #
        # REUTILIZA:
        # script_localizar_campo
        # ============================================================

        caminho_frame_autorizacao = None

        inicio = time.time()


        while time.time() - inicio < 30:

            try:

                driver.switch_to.default_content()


                caminho_frame_autorizacao = (
                    driver.execute_script(
                        script_localizar_campo,
                        xpath_campo_autorizacao
                    )
                )


                if caminho_frame_autorizacao is not None:

                    print(
                        "69 - Campo Autorização encontrado!"
                    )

                    print(
                        "Caminho dos frames:",
                        caminho_frame_autorizacao
                    )

                    break


            except Exception as erro:

                print(
                    "Aguardando campo Autorização..."
                )

                print(
                    erro
                )


            time.sleep(
                0.5
            )


        if caminho_frame_autorizacao is None:

            raise Exception(
                "Não foi possível localizar "
                "o campo para digitar 'Autorização'."
            )


        # ============================================================
        # 106. ENTRA NO FRAME CORRETO
        #
        # REUTILIZA:
        # entrar_no_caminho_frames()
        # ============================================================

        entrar_no_caminho_frames(
            caminho_frame_autorizacao
        )


        # ============================================================
        # 107. LOCALIZA O CAMPO
        # ============================================================

        campo_autorizacao = WebDriverWait(
            driver,
            20
        ).until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    xpath_campo_autorizacao
                )
            )
        )


        print(
            "70 - Campo Autorização pronto para preenchimento."
        )


        # ============================================================
        # 108. CLICA E LIMPA O CAMPO
        # ============================================================

        campo_autorizacao.click()


        campo_autorizacao.send_keys(
            Keys.CONTROL,
            "a"
        )


        campo_autorizacao.send_keys(
            Keys.BACKSPACE
        )


        time.sleep(
            0.3
        )


        # ============================================================
        # 109. DIGITA AUTORIZAÇÃO
        # ============================================================

        print("")
        print(
            "71 - Digitando 'Autorização'..."
        )


        for letra in "Autorização":

            campo_autorizacao.send_keys(
                letra
            )

            time.sleep(
                0.1
            )


        print(
            "71 - 'Autorização' digitado."
        )


        # ============================================================
        # CONFERE O VALOR DIGITADO
        # ============================================================

        valor_autorizacao = (
            campo_autorizacao.get_attribute(
                "value"
            )
        )


        print(
            "Valor atual do campo:",
            valor_autorizacao
        )


        # ============================================================
        # 110. AGUARDA O SEI PROCESSAR O TEXTO
        # ============================================================

        time.sleep(
            1
        )


        # ============================================================
        # 111. TAB + ENTER
        #
        # Primeiro envia diretamente ao campo:
        # TAB e depois ENTER
        # ============================================================

        print("")
        print(
            "72 - Pressionando TAB + ENTER..."
        )


        try:

            campo_autorizacao.send_keys(
                Keys.TAB
            )

            time.sleep(
                0.3
            )

            ActionChains(
                driver
            ).send_keys(
                Keys.ENTER
            ).perform()


            print(
                "72 - TAB + ENTER enviado com sucesso."
            )


        except Exception as erro:

            print(
                "Primeira tentativa de TAB + ENTER falhou."
            )

            print(
                erro
            )

            print(
                "Tentando novamente com ActionChains..."
            )


            entrar_no_caminho_frames(
                caminho_frame_autorizacao
            )


            campo_autorizacao = WebDriverWait(
                driver,
                10
            ).until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        xpath_campo_autorizacao
                    )
                )
            )


            ActionChains(
                driver
            ).click(
                campo_autorizacao
            ).send_keys(
                Keys.TAB
            ).pause(
                0.3
            ).send_keys(
                Keys.ENTER
            ).perform()


            print(
                "72 - TAB + ENTER enviado com ActionChains."
            )


        # ============================================================
        # 112. AGUARDA ABERTURA DA PRÓXIMA TELA
        # ============================================================

        print("")
        print(
            "73 - Aguardando abertura da próxima tela..."
        )


        time.sleep(
            3
        )


        # ============================================================
        # VOLTA PARA DOCUMENTO PRINCIPAL
        # ============================================================

        driver.switch_to.default_content()


        print("")
        print(
            "=============================================="
        )

        print(
            "AUTORIZAÇÃO SELECIONADA COM SUCESSO"
        )

        print(
            "=============================================="
        )




        # ============================================================
        # NOVA ETAPA - PREENCHER CAMPO AP COM DADOS DO DATAFRAME
        # XPATH: /html/body/div[1]/div/div/form[1]/div[6]/input
        # Coluna: AP (mesma linha do Processo SEI, NFS e CONTRATO)
        # ============================================================

        print("74 - Preparando preenchimento do campo AP após selecionar Autorização...")

        if "AP" not in df.columns:
            raise ValueError("A coluna 'AP' não foi encontrada na planilha.")

        valor_ap = df.loc[indice_linha_atual, "AP"]

        if pd.isna(valor_ap) or not str(valor_ap).strip():
            raise ValueError(
                f"A coluna AP está vazia na linha do DataFrame {indice_linha_atual}."
            )

        # Evita que números inteiros recebam sufixo .0 ao ler o Excel.
        if isinstance(valor_ap, (float, int)) and float(valor_ap).is_integer():
            texto_ap = str(int(valor_ap))
        else:
            texto_ap = str(valor_ap).strip()

        xpath_campo_ap = "/html/body/div[1]/div/div/form[1]/div[6]/input"

        caminho_frame_ap = None
        inicio_ap = time.time()

        while time.time() - inicio_ap < 30:
            driver.switch_to.default_content()
            caminho_frame_ap = driver.execute_script(
                script_localizar_campo, xpath_campo_ap
            )
            if caminho_frame_ap is not None:
                break
            time.sleep(0.5)

        if caminho_frame_ap is None:
            raise TimeoutException("Campo AP não encontrado na tela do SEI.")

        entrar_no_caminho_frames(caminho_frame_ap)

        campo_ap = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, xpath_campo_ap))
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});", campo_ap
        )
        campo_ap.click()
        campo_ap.send_keys(Keys.CONTROL, "a")
        campo_ap.send_keys(Keys.BACKSPACE)
        campo_ap.send_keys(texto_ap)

        valor_preenchido_ap = campo_ap.get_attribute("value") or ""
        if valor_preenchido_ap.strip() != texto_ap:
            raise RuntimeError(
                f"Falha na validação de AP: esperado {texto_ap!r}, "
                f"encontrado {valor_preenchido_ap!r}."
            )

        campo_ap.send_keys(Keys.TAB)
        print("75 - Campo AP preenchido e validado:", texto_ap)
        driver.switch_to.default_content()

        # ============================================================
        # 76. AUTORIZAÇÃO - PREENCHER NOME NA ÁRVORE COM A COLUNA AP
        # XPATH: /html/body/div[1]/div/div/form[1]/div[7]/input[2]
        # Reutiliza texto_ap da mesma linha do Processo SEI.
        # Não altera o Nome na Árvore da Nota Fiscal (CONTRATO).
        # ============================================================

        print("76 - Procurando Nome na Árvore da Autorização...")

        xpath_nome_arvore_autorizacao = (
            "/html/body/div[1]/div/div/form[1]/div[7]/input[2]"
        )

        caminho_frame_nome_arvore_autorizacao = None
        inicio_nome_arvore_autorizacao = time.time()

        while time.time() - inicio_nome_arvore_autorizacao < 30:
            driver.switch_to.default_content()
            caminho_frame_nome_arvore_autorizacao = driver.execute_script(
                script_localizar_campo, xpath_nome_arvore_autorizacao
            )
            if caminho_frame_nome_arvore_autorizacao is not None:
                break
            time.sleep(0.5)

        if caminho_frame_nome_arvore_autorizacao is None:
            raise TimeoutException(
                "Campo Nome na Árvore da Autorização não encontrado."
            )

        entrar_no_caminho_frames(caminho_frame_nome_arvore_autorizacao)

        campo_nome_arvore_autorizacao = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable(
                (By.XPATH, xpath_nome_arvore_autorizacao)
            )
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            campo_nome_arvore_autorizacao,
        )

        campo_nome_arvore_autorizacao.click()
        campo_nome_arvore_autorizacao.send_keys(Keys.CONTROL, "a")
        campo_nome_arvore_autorizacao.send_keys(Keys.BACKSPACE)
        campo_nome_arvore_autorizacao.send_keys(texto_ap)

        valor_nome_arvore_autorizacao = (
            campo_nome_arvore_autorizacao.get_attribute("value") or ""
        )

        if valor_nome_arvore_autorizacao.strip() != texto_ap:
            raise RuntimeError(
                "Falha ao preencher Nome na Árvore da Autorização: "
                f"esperado {texto_ap!r}, "
                f"encontrado {valor_nome_arvore_autorizacao!r}."
            )

        campo_nome_arvore_autorizacao.send_keys(Keys.TAB)
        print("77 - Nome na Árvore da Autorização preenchido com AP:", texto_ap)

        driver.switch_to.default_content()

        # ============================================================
        # 78. AUTORIZAÇÃO - NÍVEL DE ACESSO: PÚBLICO
        # XPath fornecido: /html/body/div[1]/div/div/form[1]/div[15]/fieldset/div[3]/div/label
        # ============================================================

        print('78 - Procurando Nível de Acesso da Autorização...')
        xpath_publico_autorizacao = (
            '/html/body/div[1]/div/div/form[1]/div[15]/fieldset/div[3]/div/label'
        )

        caminho_frame_publico_autorizacao = None
        inicio_publico_autorizacao = time.time()
        while time.time() - inicio_publico_autorizacao < 30:
            driver.switch_to.default_content()
            caminho_frame_publico_autorizacao = driver.execute_script(
                script_localizar_campo, xpath_publico_autorizacao
            )
            if caminho_frame_publico_autorizacao is not None:
                break
            time.sleep(0.5)

        if caminho_frame_publico_autorizacao is None:
            raise TimeoutException('Nível de Acesso da Autorização não encontrado.')

        entrar_no_caminho_frames(caminho_frame_publico_autorizacao)
        label_publico_autorizacao = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, xpath_publico_autorizacao))
        )

        # O .text do Selenium pode estar vazio em labels estilizados.
        # Examina o textContent e o contêiner da opção também.
        info_opcao = driver.execute_script("""
        const label = arguments[0];
        const container = label.closest('div') || label.parentElement;
        let radio = null;
        if (label.htmlFor) radio = label.ownerDocument.getElementById(label.htmlFor);
        if (!radio) radio = label.querySelector('input[type="radio"]');
        if (!radio && container) radio = container.querySelector('input[type="radio"]');
        return {
            textoLabel: (label.textContent || '').trim(),
            textoContainer: (container?.textContent || '').trim(),
            radioEncontrado: !!radio,
            radioValor: radio?.value || '',
            radioId: radio?.id || ''
        };
        """, label_publico_autorizacao)
        texto_opcao = (label_publico_autorizacao.text or '').strip()
        texto_referencia = ' '.join([
            texto_opcao, info_opcao['textoLabel'], info_opcao['textoContainer']
        ]).casefold()
        print('79 - Dados da opção de nível de acesso:', info_opcao)
        # Texto vazio não invalida o XPath: alguns labels não exibem texto próprio.
        if texto_referencia.strip() and not any(
            x in texto_referencia for x in ('público', 'publico')
        ):
            raise RuntimeError(
                'A opção localizada exibe um nível diferente de Público. '
                f'Detalhes: {info_opcao}'
            )
        if not texto_referencia.strip():
            print('79 - Aviso: opção sem texto; será verificado o radio associado ao XPath informado.')

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            label_publico_autorizacao
        )
        try:
            label_publico_autorizacao.click()
        except WebDriverException:
            driver.execute_script('arguments[0].click();', label_publico_autorizacao)

        resultado_publico_autorizacao = driver.execute_script("""
        const label = arguments[0];
        const container = label.closest('div') || label.parentElement;
        let radio = null;
        if (label.htmlFor) radio = label.ownerDocument.getElementById(label.htmlFor);
        if (!radio) radio = label.querySelector('input[type="radio"]');
        if (!radio && container) radio = container.querySelector('input[type="radio"]');
        return {
            encontrou: !!radio,
            marcado: radio ? radio.checked : null,
            valor: radio?.value || null,
            texto: (container?.textContent || label.textContent || '').trim()
        };
        """, label_publico_autorizacao)
        if not resultado_publico_autorizacao['encontrou'] or not resultado_publico_autorizacao['marcado']:
            raise RuntimeError(
                'Não foi possível confirmar a seleção do radio de Nível de Acesso; '
                f'salvamento cancelado. Detalhes: {resultado_publico_autorizacao}'
            )
        print('80 - Radio correspondente ao XPath selecionado e validado.')

        # ============================================================
        # 81. AUTORIZAÇÃO - SALVAR
        # XPath fornecido: /html/body/div[1]/div/div/div[2]/button[1]
        # ============================================================

        print('81 - Procurando botão Salvar da Autorização...')
        xpath_salvar_autorizacao = '/html/body/div[1]/div/div/div[2]/button[1]'
        caminho_frame_salvar_autorizacao = None
        inicio_salvar_autorizacao = time.time()
        while time.time() - inicio_salvar_autorizacao < 30:
            driver.switch_to.default_content()
            caminho_frame_salvar_autorizacao = driver.execute_script(
                script_localizar_campo, xpath_salvar_autorizacao
            )
            if caminho_frame_salvar_autorizacao is not None:
                break
            time.sleep(0.5)

        if caminho_frame_salvar_autorizacao is None:
            raise TimeoutException('Botão Salvar da Autorização não encontrado.')

        entrar_no_caminho_frames(caminho_frame_salvar_autorizacao)
        botao_salvar_autorizacao = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, xpath_salvar_autorizacao))
        )

        rotulo_salvar = (botao_salvar_autorizacao.text or '').strip()
        texto_botao = ' '.join([
            rotulo_salvar,
            botao_salvar_autorizacao.get_attribute('title') or '',
            botao_salvar_autorizacao.get_attribute('aria-label') or '',
            botao_salvar_autorizacao.get_attribute('textContent') or ''
        ]).strip()
        if texto_botao and 'salvar' not in texto_botao.casefold():
            raise RuntimeError(
                f'O XPath não parece apontar para o botão Salvar: {texto_botao!r}.'
            )
        if not texto_botao:
            print('81 - Botão sem texto; usando o XPath informado e validando que está clicável.')

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            botao_salvar_autorizacao
        )
        try:
            botao_salvar_autorizacao.click()
        except WebDriverException:
            driver.execute_script('arguments[0].click();', botao_salvar_autorizacao)

        print('82 - Clique em Salvar da Autorização realizado.')
        # Um alerta nativo bloqueia comandos Selenium até ser resolvido.
        # Confirma SOMENTE o alerta de duplicidade previamente previsto.
        confirmar_alerta_duplicidade(timeout=5)
        time.sleep(3)
        confirmar_alerta_duplicidade(timeout=2)
        driver.switch_to.default_content()
        print('83 - Fluxo de salvamento da Autorização concluído (verifique o documento no SEI).')


        # ============================================================
        # 84. EDITOR DA AUTORIZAÇÃO - LOCALIZAR NOVA JANELA
        # Estratégia: alterar APENAS o marcador usando a API CKEditor,
        # se disponível; caso contrário, editar o nó de texto no iframe.
        # Não utiliza Ctrl+A nem depende de clique por coordenadas.
        # ============================================================

        print('84 - Procurando nova janela do editor de Autorização...')

        TEXTO_MARCADOR = '[digite aqui o texto do conteúdo principal do documento]'

        # Aguarda a janela com a URL típica do editor do SEI.
        janela_editor = None
        inicio_editor = time.time()
        while time.time() - inicio_editor < 60:
            for handle in driver.window_handles:
                try:
                    driver.switch_to.window(handle)
                    driver.switch_to.default_content()
                    url = driver.current_url.lower()
                    if 'editor_montar' in url:
                        janela_editor = handle
                        break
                except WebDriverException:
                    pass
            if janela_editor is not None:
                break
            time.sleep(0.5)

        if janela_editor is None:
            raise RuntimeError(
                'Janela editor_montar não encontrada. Verifique se o SEI abriu o editor.'
            )

        print('85 - Janela do editor encontrada:', driver.current_url)

        # ============================================================
        # 86-88. LIMPAR TODO O CONTEÚDO EDITÁVEL DA AUTORIZAÇÃO
        #        Ctrl+A seguido de Delete, ANTES de inserir a imagem.
        #        A seleção é restrita ao corpo do CKEditor, nunca à página inteira.
        # ============================================================
        from selenium.webdriver.common.keys import Keys
        from selenium.webdriver.common.action_chains import ActionChains

        print('86 - Localizando o corpo editável da Autorização para Ctrl+A e Delete...')
        driver.switch_to.window(janela_editor)
        driver.switch_to.default_content()

        SCRIPT_LOCALIZAR_EDITOR_AP = r"""
        const ck = window.CKEDITOR;
        if (!ck || !ck.instances) return null;
        const editors = Object.values(ck.instances).filter(
            ed => ed.status === 'ready' && ed.editable && ed.editable()
        );
        if (!editors.length) return null;
        const marcador = '[digite aqui o texto do conteúdo principal do documento]';
        // Quando há mais de um editor na mesma janela, prioriza aquele com o marcador.
        const alvo = editors.find(ed => {
            try { return (ed.editable().getText() || '').includes(marcador); }
            catch(e) { return false; }
        }) || editors.find(ed => {
            try { return ed.editable().isVisible(); } catch(e) { return false; }
        }) || editors[0];
        try {
            alvo.focus();
            const edEl = alvo.editable().$;
            const doc = edEl.ownerDocument;
            const iframe = doc.defaultView && doc.defaultView.frameElement;
            return {name: alvo.name, frame: iframe || null, inline: !iframe};
        } catch(e) { return null; }
        """

        info_editor_ap = None
        fim_espera_editor_ap = time.monotonic() + 30
        while time.monotonic() < fim_espera_editor_ap:
            try:
                info_editor_ap = driver.execute_script(SCRIPT_LOCALIZAR_EDITOR_AP)
                if info_editor_ap:
                    break
            except WebDriverException:
                pass
            time.sleep(0.5)

        if not info_editor_ap:
            raise TimeoutException('CKEditor da Autorização não ficou pronto para limpeza.')

        if info_editor_ap.get('frame') is not None:
            driver.switch_to.frame(info_editor_ap['frame'])
            alvo_teclado_ap = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'body.cke_editable, body[contenteditable="true"], body'))
            )
            if not driver.execute_script('return document.designMode === "on" || arguments[0].isContentEditable;', alvo_teclado_ap):
                raise RuntimeError('O corpo identificado não está editável; Ctrl+A cancelado.')
        else:
            alvo_teclado_ap = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[contenteditable="true"].cke_editable, .cke_editable[contenteditable="true"]'))
            )

        alvo_teclado_ap.click()
        ActionChains(driver).key_down(Keys.CONTROL).send_keys('a').key_up(Keys.CONTROL).send_keys(Keys.DELETE).perform()
        time.sleep(0.8)
        driver.switch_to.default_content()
        print('87 - Ctrl+A e Delete enviados ao conteúdo editável:', info_editor_ap['name'])
        print('88 - Conteúdo da Autorização limpo. Prosseguindo para inserir a imagem.')


        # ============================================================
        # 89. EDITOR DA AUTORIZAÇÃO - CLICAR NO BOTÃO PELO XPATH
        # Janela: editor_montar (aberta na etapa anterior)
        # ============================================================

        print("89 - Localizando o botão solicitado na janela do editor...")

        XPATH_BOTAO_EDITOR = (
            "/html/body/form/div[1]/div[3]/div/div/"
            "span[2]/span[5]/span[3]/a[5]/span[1]"
        )

        # A etapa anterior pode ter deixado o Selenium dentro de um iframe.
        driver.switch_to.window(janela_editor)
        driver.switch_to.default_content()

        # Primeiro procura na página principal. Se necessário, percorre frames.
        def localizar_xpath_no_editor(xpath, profundidade=0):
            try:
                elementos = driver.find_elements(By.XPATH, xpath)
                if elementos:
                    return True
            except WebDriverException:
                pass

            if profundidade >= 8:
                return False

            frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
            for indice in range(len(frames)):
                try:
                    # Renova referências para evitar elementos obsoletos.
                    frames_atuais = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
                    driver.switch_to.frame(frames_atuais[indice])
                    if localizar_xpath_no_editor(xpath, profundidade + 1):
                        return True  # Permanece no frame correto.
                    driver.switch_to.parent_frame()
                except WebDriverException:
                    try:
                        driver.switch_to.parent_frame()
                    except WebDriverException:
                        pass
            return False

        prazo_botao = time.monotonic() + 30
        botao_editor_encontrado = False
        while time.monotonic() < prazo_botao:
            driver.switch_to.default_content()
            if localizar_xpath_no_editor(XPATH_BOTAO_EDITOR):
                botao_editor_encontrado = True
                break
            time.sleep(0.5)

        if not botao_editor_encontrado:
            raise TimeoutException(
                "O botão do XPath informado não apareceu na janela do editor em 30 segundos."
            )

        # O XPath aponta para um SPAN dentro de um link <a>.
        # O elemento pode existir no DOM sem satisfazer element_to_be_clickable.
        # Conserva o frame onde a busca acima encontrou o elemento.
        botao_editor = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, XPATH_BOTAO_EDITOR))
        )

        # Aciona o link pai, que normalmente é o controle real da barra.
        link_editor = driver.execute_script(
            "return arguments[0].closest('a') || arguments[0];",
            botao_editor
        )

        informacoes_botao = driver.execute_script("""
            const el = arguments[0];
            return {
                tag: el.tagName,
                titulo: el.getAttribute('title') || '',
                texto: (el.textContent || '').trim().slice(0, 100),
                ariaDisabled: el.getAttribute('aria-disabled'),
                desabilitado: el.hasAttribute('disabled'),
                visivel: !!(el.getClientRects().length)
            };
        """, link_editor)
        print('89 - Diagnóstico do botão:', informacoes_botao)

        if informacoes_botao['desabilitado'] or informacoes_botao['ariaDisabled'] == 'true':
            raise RuntimeError('O botão foi encontrado, mas está desabilitado.')

        # Tenta o clique convencional; para controles que não são considerados
        # visíveis pelo Selenium, usa o clique JS no mesmo link encontrado.
        try:
            if link_editor.is_displayed():
                driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center',inline:'center'});",
                    link_editor
                )
                link_editor.click()
            else:
                driver.execute_script('arguments[0].click();', link_editor)
        except WebDriverException as erro_clique:
            print('89 - Clique convencional falhou:', erro_clique)
            driver.execute_script('arguments[0].click();', link_editor)

        print('90 - Comando de clique enviado ao link associado ao XPath informado.')


        # ============================================================
        # 91. CONTINUAÇÃO - CLICAR EM /html/body/form/input
        # O código anterior é mantido integralmente.
        # ============================================================

        print("91 - Procurando o campo /html/body/form/input...")
        XPATH_INPUT_FINAL = "/html/body/form/input"


        def localizar_input_final_no_contexto(profundidade=0):
            """Localiza o XPath na página ou em frames; mantém o frame encontrado."""
            try:
                candidatos = driver.find_elements(By.XPATH, XPATH_INPUT_FINAL)
                if candidatos:
                    return candidatos[0]
            except WebDriverException:
                pass

            if profundidade >= 8:
                return None

            try:
                quantidade = len(driver.find_elements(By.CSS_SELECTOR, "iframe, frame"))
            except WebDriverException:
                return None

            for i in range(quantidade):
                try:
                    frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
                    driver.switch_to.frame(frames[i])
                    elemento = localizar_input_final_no_contexto(profundidade + 1)
                    if elemento is not None:
                        return elemento
                    driver.switch_to.parent_frame()
                except WebDriverException:
                    try:
                        driver.switch_to.parent_frame()
                    except WebDriverException:
                        pass
            return None


        # Começa pela janela em uso. Se o clique anterior abriu outra janela,
        # também a pesquisa, sem alterar o comportamento das etapas anteriores.
        janela_antes_do_clique = driver.current_window_handle
        input_final = None
        janela_input_final = None
        limite_input = time.monotonic() + 45

        while time.monotonic() < limite_input and input_final is None:
            janelas = driver.window_handles
            # Prioriza a janela atualmente selecionada; em seguida as mais recentes.
            ordem = [janela_antes_do_clique] + [
                h for h in reversed(janelas) if h != janela_antes_do_clique
            ]
            for janela in ordem:
                if janela not in janelas:
                    continue
                try:
                    driver.switch_to.window(janela)
                    driver.switch_to.default_content()
                    input_final = localizar_input_final_no_contexto()
                    if input_final is not None:
                        janela_input_final = janela
                        break
                except WebDriverException:
                    continue
            if input_final is None:
                time.sleep(0.5)

        if input_final is None:
            raise TimeoutException(
                "O campo /html/body/form/input não apareceu em nenhuma janela/frame "
                "acessível em 45 segundos. Nenhum clique foi executado."
            )

        # Neste momento o Selenium está no frame correto do elemento localizado.
        dados_input_final = driver.execute_script("""
            const el = arguments[0];
            return {
                tag: el.tagName,
                tipo: el.getAttribute('type') || '',
                valor: el.getAttribute('value') || '',
                desabilitado: !!el.disabled,
                visivel: !!el.getClientRects().length
            };
        """, input_final)
        print("91 - Input encontrado:", dados_input_final)

        if dados_input_final["desabilitado"]:
            raise RuntimeError("O input final foi encontrado, mas está desabilitado.")

        try:
            if input_final.is_displayed():
                driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center',inline:'center'});",
                    input_final
                )
                input_final.click()
                print("92 - Input clicado pelo Selenium.")
            else:
                driver.execute_script("arguments[0].click();", input_final)
                print("92 - Input clicado via JavaScript (elemento não visível).")
        except WebDriverException as erro_input_final:
            print("92 - Clique normal falhou:", erro_input_final)
            driver.execute_script("arguments[0].click();", input_final)
            print("92 - Input clicado via JavaScript.")

        print("93 - Clique em /html/body/form/input enviado.")


        # ============================================================
        # 94. CKEDITOR - ACIONAR "ESCOLHER ARQUIVO" (JANELA IMAGEM)
        #     Acrescentado ao final; etapas anteriores preservadas.
        # ============================================================

        print("94 - Localizando Escolher arquivo na janela Imagem...")

        # Se quiser enviar o arquivo diretamente, configure aqui um caminho
        # absoluto de imagem (ex.: r"C:\\Imagens\\logotipo.png").
        # Deixe como None para abrir a janela de escolha manual do Windows.
        CAMINHO_IMAGEM_EDITOR = None


        def localizar_input_imagem(profundidade=0):
            """Procura o input file do diálogo Imagem e mantém o frame correto."""
            try:
                arquivos = driver.find_elements(By.CSS_SELECTOR, "input[type='file']")
                for campo in arquivos:
                    if campo.is_enabled():
                        return campo
            except WebDriverException:
                pass

            if profundidade >= 8:
                return None

            try:
                quantidade = len(driver.find_elements(By.CSS_SELECTOR, "iframe, frame"))
            except WebDriverException:
                return None

            for indice in range(quantidade):
                try:
                    frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
                    driver.switch_to.frame(frames[indice])
                    localizado = localizar_input_imagem(profundidade + 1)
                    if localizado is not None:
                        return localizado
                    driver.switch_to.parent_frame()
                except WebDriverException:
                    try:
                        driver.switch_to.parent_frame()
                    except WebDriverException:
                        pass
            return None


        # O diálogo CKEditor pode estar na janela principal do editor,
        # mesmo que a etapa anterior tenha interagido com outro iframe.
        janela_atual_imagem = driver.current_window_handle
        campo_escolher_arquivo = None
        limite_escolher_arquivo = time.monotonic() + 40

        while time.monotonic() < limite_escolher_arquivo:
            handles = driver.window_handles
            ordem = [janela_atual_imagem] + [h for h in reversed(handles) if h != janela_atual_imagem]
            for handle in ordem:
                if handle not in handles:
                    continue
                try:
                    driver.switch_to.window(handle)
                    driver.switch_to.default_content()
                    # A classe "cke_dialog_ui_input_file" identifica o iframe
                    # do CKEditor; o ID cke_460_fileInput pode variar.
                    iframes = driver.find_elements(
                        By.CSS_SELECTOR, "iframe.cke_dialog_ui_input_file"
                    )
                    for iframe in iframes:
                        try:
                            driver.switch_to.frame(iframe)
                            campos = driver.find_elements(By.CSS_SELECTOR, "input[type='file']")
                            if campos and campos[0].is_enabled():
                                campo_escolher_arquivo = campos[0]
                                break
                            driver.switch_to.parent_frame()
                        except WebDriverException:
                            driver.switch_to.default_content()
                    if campo_escolher_arquivo is None:
                        driver.switch_to.default_content()
                        campo_escolher_arquivo = localizar_input_imagem()
                    if campo_escolher_arquivo is not None:
                        break
                except WebDriverException:
                    continue
            if campo_escolher_arquivo is not None:
                break
            time.sleep(0.5)

        if campo_escolher_arquivo is None:
            raise TimeoutException(
                "Não foi encontrado o input de arquivo do diálogo Imagem do CKEditor."
            )

        # ============================================================
        # 95. IMAGEM DA COLUNA "Autorização de Pagamento" DO DATAFRAME
        #     Usa a MESMA linha do Processo SEI já selecionado.
        #     Aceita caminho absoluto ou nome/caminho relativo do arquivo.
        # ============================================================

        from pathlib import Path
        import os
        import re
        import unicodedata
        from selenium.common.exceptions import StaleElementReferenceException

        COLUNA_IMAGEM_AP = "Autorização de Pagamento"
        PASTA_IMAGENS_AP = Path(r"X:\GOB\Teste Automação - NÃO DELETAR")
        EXTENSOES_IMAGEM_AP = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp")


        def _normalizar_nome_imagem_ap(nome):
            """Ignora diferenças de Unicode, espaços, acentos e letras maiúsculas."""
            nome = unicodedata.normalize("NFKC", str(nome)).strip()
            nome = re.sub(r"\s+", " ", nome)
            nome = unicodedata.normalize("NFKD", nome)
            nome = "".join(c for c in nome if not unicodedata.combining(c))
            return nome.casefold()


        def _localizar_arquivo_ap(valor_excel, pasta_base):
            """Não escolhe arquivos ambíguos: só aceita correspondência inequívoca."""
            valor = str(valor_excel).strip().strip('"').strip("'")
            valor = unicodedata.normalize("NFKC", valor)
            if not valor:
                raise ValueError("Nome de imagem vazio na coluna Autorização de Pagamento.")

            referencia = Path(os.path.expandvars(os.path.expanduser(valor)))
            if referencia.is_absolute():
                if referencia.is_file() and referencia.suffix.lower() in EXTENSOES_IMAGEM_AP:
                    return referencia.resolve()
                diretorio_busca = referencia.parent
                nome_busca = referencia.name
            else:
                # Nome simples ou caminho relativo dentro da pasta configurada.
                candidato = pasta_base / referencia
                if candidato.is_file() and candidato.suffix.lower() in EXTENSOES_IMAGEM_AP:
                    return candidato.resolve()
                diretorio_busca = pasta_base / referencia.parent
                nome_busca = referencia.name

            if not diretorio_busca.is_dir():
                raise FileNotFoundError(f"Pasta de imagens não acessível: {diretorio_busca}")

            chave = _normalizar_nome_imagem_ap(nome_busca)
            sem_extensao = Path(nome_busca).suffix == ""
            iguais = []

            # Primeiro procura somente na pasta esperada, usando nomes normalizados.
            for arquivo in diretorio_busca.iterdir():
                if not arquivo.is_file() or arquivo.suffix.lower() not in EXTENSOES_IMAGEM_AP:
                    continue
                comparacao = arquivo.stem if sem_extensao else arquivo.name
                if _normalizar_nome_imagem_ap(comparacao) == chave:
                    iguais.append(arquivo)

            # Se a planilha traz somente o nome, permite localizar em subpastas.
            if not iguais and not referencia.is_absolute() and referencia.parent == Path('.'):
                print("95.1 - Imagem não encontrada na pasta raiz; pesquisando subpastas...")
                for raiz, _, arquivos in os.walk(pasta_base):
                    if Path(raiz) == pasta_base:
                        continue
                    for nome in arquivos:
                        if Path(nome).suffix.lower() not in EXTENSOES_IMAGEM_AP:
                            continue
                        comparacao = Path(nome).stem if sem_extensao else nome
                        if _normalizar_nome_imagem_ap(comparacao) == chave:
                            iguais.append(Path(raiz) / nome)

            if len(iguais) > 1:
                raise RuntimeError(
                    "Há mais de uma imagem que corresponde ao nome informado no Excel; "
                    "não é seguro escolher uma arbitrariamente:\n" +
                    "\n".join(str(p) for p in iguais)
                )
            if not iguais:
                raise FileNotFoundError(
                    f"Imagem não encontrada para {valor!r}, nem com normalização de espaços "
                    f"e letras. Pasta pesquisada: {diretorio_busca}"
                )
            return iguais[0].resolve()


        if COLUNA_IMAGEM_AP not in df.columns:
            raise KeyError(f"Coluna obrigatória ausente: {COLUNA_IMAGEM_AP!r}")

        valor_imagem_ap = df.loc[indice_linha_atual, COLUNA_IMAGEM_AP]
        if pd.isna(valor_imagem_ap) or not str(valor_imagem_ap).strip():
            raise ValueError(
                f"A coluna {COLUNA_IMAGEM_AP!r} está vazia na linha "
                f"{indice_linha_atual} (Processo SEI: {processo_sei})."
            )

        arquivo_imagem_ap = _localizar_arquivo_ap(valor_imagem_ap, PASTA_IMAGENS_AP)
        if arquivo_imagem_ap.stat().st_size == 0:
            raise ValueError(f"A imagem está vazia: {arquivo_imagem_ap}")
        print("95 - Imagem localizada:", arquivo_imagem_ap)

        # Mesmo quando o arquivo existe, o CKEditor pode recriar o iframe do campo
        # de upload. Para evitar StaleElementReferenceException, reobtém o input.
        # A seleção de arquivo é verificada antes de clicar em OK.
        janela_upload_ap = driver.current_window_handle
        CAMINHO_UPLOAD_AP = str(arquivo_imagem_ap)
        MAX_TENTATIVAS_UPLOAD_AP = 3
        TEMPO_ESPERA_UPLOAD_AP = 5


        def _obter_input_upload_ap():
            driver.switch_to.window(janela_upload_ap)
            driver.switch_to.default_content()
            for iframe in driver.find_elements(By.CSS_SELECTOR, "iframe.cke_dialog_ui_input_file"):
                try:
                    if not iframe.is_displayed():
                        continue
                    driver.switch_to.frame(iframe)
                    inputs = driver.find_elements(By.CSS_SELECTOR, 'input[type="file"]')
                    if inputs and inputs[0].is_enabled():
                        return inputs[0]  # mantém o contexto dentro do iframe
                except (StaleElementReferenceException, WebDriverException):
                    pass
                driver.switch_to.default_content()
            return None


        def _arquivo_ap_selecionado_no_input(campo, esperado):
            try:
                valor_input = campo.get_attribute("value") or ""
                # O navegador retorna C:\fakepath\nome.jpg por privacidade.
                nome_input = valor_input.replace("/", "\\").split("\\")[-1]
                return _normalizar_nome_imagem_ap(nome_input) == _normalizar_nome_imagem_ap(esperado)
            except (StaleElementReferenceException, WebDriverException):
                return False


        upload_confirmado_ap = False
        for tentativa_ap in range(1, MAX_TENTATIVAS_UPLOAD_AP + 1):
            print(f"96 - Tentativa {tentativa_ap}/{MAX_TENTATIVAS_UPLOAD_AP} de selecionar imagem...")
            campo_upload_ap = None
            prazo_campo_ap = time.monotonic() + 15
            while time.monotonic() < prazo_campo_ap:
                try:
                    campo_upload_ap = _obter_input_upload_ap()
                    if campo_upload_ap is not None:
                        break
                except (StaleElementReferenceException, WebDriverException):
                    pass
                time.sleep(0.5)
            if campo_upload_ap is None:
                if tentativa_ap == MAX_TENTATIVAS_UPLOAD_AP:
                    raise TimeoutException("Campo de upload de imagem não encontrado no CKEditor.")
                time.sleep(2)
                continue

            try:
                # Não clicar no input[type=file]; apenas passar caminho completo.
                campo_upload_ap.send_keys(CAMINHO_UPLOAD_AP)
                print("96.1 - Caminho da imagem enviado ao campo:", CAMINHO_UPLOAD_AP)
                # Valida imediatamente: alguns uploads limpam o input após envio.
                upload_confirmado_ap = _arquivo_ap_selecionado_no_input(
                    campo_upload_ap, arquivo_imagem_ap.name
                )
                if not upload_confirmado_ap:
                    # Pode ter sido substituído pelo editor após o evento change.
                    # Aguarda estabilizar e tenta validar o novo input.
                    time.sleep(1)
                    campo_novo_ap = _obter_input_upload_ap()
                    if campo_novo_ap is not None:
                        upload_confirmado_ap = _arquivo_ap_selecionado_no_input(
                            campo_novo_ap, arquivo_imagem_ap.name
                        )
                if upload_confirmado_ap:
                    break
                print("96.2 - Campo não confirmou o nome do arquivo; nova tentativa.")
            except (StaleElementReferenceException, WebDriverException) as erro_ap:
                print(f"96.2 - Falha transitória na seleção: {type(erro_ap).__name__}")
            time.sleep(2)

        if not upload_confirmado_ap:
            raise RuntimeError(
                "Não foi possível confirmar a seleção do arquivo no campo do CKEditor. "
                "A automação foi interrompida antes do OK para evitar documento sem imagem. "
                f"Arquivo esperado: {CAMINHO_UPLOAD_AP}"
            )

        print("97 - Nome da imagem confirmado no campo do CKEditor.")
        print(f"97.1 - Aguardando {TEMPO_ESPERA_UPLOAD_AP} segundos antes do botão OK...")
        time.sleep(TEMPO_ESPERA_UPLOAD_AP)
        print("98 - Preparando confirmação da imagem no diálogo.")

        # ============================================================
        # 98. CLICA EM OK NO DIÁLOGO "Imagem" DO CKEDITOR
        #     Mantém a mesma janela/aba do editor já em uso.
        # ============================================================

        print("98 - Procurando botão OK do diálogo Imagem...")

        # Depois do send_keys no input[type='file'], o botão OK fica no documento
        # principal da mesma janela do editor, fora do iframe do campo de arquivo.
        driver.switch_to.default_content()

        botao_ok_imagem = None
        limite_botao_ok = time.monotonic() + 30

        while time.monotonic() < limite_botao_ok:
            try:
                driver.switch_to.default_content()

                candidatos_ok = driver.find_elements(
                    By.CSS_SELECTOR,
                    "a.cke_dialog_ui_button_ok, #cke_467_uiElement"
                )

                for candidato in candidatos_ok:
                    try:
                        if candidato.is_displayed():
                            botao_ok_imagem = candidato
                            break
                    except WebDriverException:
                        continue

                if botao_ok_imagem is None:
                    candidatos_ok = driver.find_elements(
                        By.XPATH,
                        "//a[contains(@class,'cke_dialog_ui_button_ok') or @title='OK' or .//span[normalize-space()='OK']]"
                    )
                    for candidato in candidatos_ok:
                        try:
                            if candidato.is_displayed():
                                botao_ok_imagem = candidato
                                break
                        except WebDriverException:
                            continue

                if botao_ok_imagem is not None:
                    break
            except WebDriverException:
                pass

            time.sleep(0.5)

        if botao_ok_imagem is None:
            raise TimeoutException(
                'Não foi possível localizar o botão OK do diálogo Imagem do CKEditor.'
            )

        try:
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center', inline:'center'});",
                botao_ok_imagem
            )
        except WebDriverException:
            pass

        time.sleep(0.3)

        clicou_ok = False

        try:
            botao_ok_imagem.click()
            clicou_ok = True
            print('99 - Botão OK clicado com Selenium.')
        except WebDriverException as erro_ok:
            print('Clique normal no botão OK falhou:', erro_ok)
            try:
                driver.execute_script("arguments[0].click();", botao_ok_imagem)
                clicou_ok = True
                print('99 - Botão OK clicado via JavaScript.')
            except WebDriverException as erro_ok_js:
                print('Clique via JavaScript no botão OK falhou:', erro_ok_js)

        if not clicou_ok:
            raise RuntimeError('Não foi possível clicar no botão OK do diálogo Imagem.')

        print('100 - Imagem confirmada no diálogo do CKEditor.')


        # ============================================================
        # 101. SALVAR DOCUMENTO APÓS CONFIRMAR IMAGEM NO CKEDITOR
        # ============================================================

        print("101 - Procurando o botão Salvar do CKEditor...")
        driver.switch_to.default_content()

        # Aguarda o diálogo Imagem ser fechado antes de salvar.
        WebDriverWait(driver, 30).until(
            lambda d: not any(
                dialog.is_displayed()
                for dialog in d.find_elements(By.CSS_SELECTOR, ".cke_dialog_container")
            )
        )

        salvou_editor = False
        prazo_salvar = time.monotonic() + 30

        while time.monotonic() < prazo_salvar and not salvou_editor:
            driver.switch_to.default_content()

            # Não usar IDs dinâmicos como cke_27 ou cke_87.
            botoes_salvar = driver.find_elements(
                By.CSS_SELECTOR, "a.cke_button__save"
            )
            for botao_salvar in botoes_salvar:
                try:
                    if (not botao_salvar.is_displayed()
                            or botao_salvar.get_attribute("aria-disabled") == "true"
                            or "cke_button_disabled" in (botao_salvar.get_attribute("class") or "")):
                        continue
                    try:
                        botao_salvar.click()
                    except WebDriverException:
                        driver.execute_script("arguments[0].click();", botao_salvar)
                    salvou_editor = True
                    print("102 - Clique em Salvar realizado no CKEditor.")
                    break
                except (StaleElementReferenceException, WebDriverException):
                    continue

            if not salvou_editor:
                time.sleep(0.5)

        if not salvou_editor:
            raise TimeoutException(
                "O botão Salvar do CKEditor não ficou visível e habilitado. "
                "Confira a janela do editor e se a imagem foi inserida após o OK."
            )

        # Uma ação de salvar pode ainda depender de resposta do SEI; não assumir
        # persistência apenas por ter realizado o clique.
        print("103 - Comando Salvar enviado. Confira o retorno do SEI.")


        # ============================================================
        # 104. FECHAR SOMENTE A JANELA DO EDITOR DA AUTORIZAÇÃO
        # ============================================================

        print("104 - Aguardando retorno do comando Salvar...")
        time.sleep(3)

        # O X mostrado na imagem pertence à janela do Chrome, não ao CKEditor.
        # Fecha apenas a janela secundária do editor e mantém o SEI aberto.
        driver.switch_to.default_content()
        janela_editor_para_fechar = driver.current_window_handle
        url_editor_para_fechar = driver.current_url.lower()

        if "editor_montar" not in url_editor_para_fechar:
            raise RuntimeError(
                "A janela atual não é o editor da Autorização (editor_montar). "
                "Fechamento cancelado para preservar a sessão do SEI. "
                f"URL atual: {driver.current_url}"
            )

        outras_janelas_sei = [
            identificador for identificador in driver.window_handles
            if identificador != janela_editor_para_fechar
        ]
        if not outras_janelas_sei:
            raise RuntimeError(
                "Não foi encontrada outra janela do Chrome aberta. "
                "Fechamento cancelado para evitar encerrar a sessão."
            )

        driver.close()
        print("105 - Janela do editor da Autorização fechada.")

        # Retorna preferencialmente à janela do processo SEI já identificada.
        if "janela_sei" in globals() and janela_sei in driver.window_handles:
            driver.switch_to.window(janela_sei)
        else:
            driver.switch_to.window(outras_janelas_sei[0])

        driver.switch_to.default_content()
        print("106 - Retorno à janela principal do SEI concluído.")

        # Aguarda 5 segundos para o ícone ficar visível após fechar o editor.
        print("106.1 - Aguardando 5 segundos para o ícone ficar visível...")
        time.sleep(5)


        # ============================================================
        # 107. CLICAR NO ÍCONE DA BARRA DO PROCESSO (a[9]/img)
        #      Após fechar a janela do editor da Autorização.
        # ============================================================

        XPATH_ICONE_PROCESSO = "/html/body/div[1]/div/div/div[2]/div/a[9]/img"
        print("107 - Procurando o ícone informado na janela principal do SEI...")

        # Procura o XPath no documento principal ou em frames aninhados.
        # Retorna a sequência de índices para entrar no frame correto.
        SCRIPT_BUSCAR_ICONE_PROCESSO = r"""
        const xpath = arguments[0];
        function procurar(win, caminho) {
            let doc;
            try { doc = win.document; } catch (e) { return null; }
            try {
                const achado = doc.evaluate(
                    xpath, doc, null, win.XPathResult.FIRST_ORDERED_NODE_TYPE, null
                ).singleNodeValue;
                if (achado) return caminho;
            } catch (e) { /* tenta os frames */ }
            for (let i = 0; i < win.frames.length; i++) {
                try {
                    const resultado = procurar(win.frames[i], caminho.concat(i));
                    if (resultado !== null) return resultado;
                } catch (e) { /* frame inacessível */ }
            }
            return null;
        }
        return procurar(window, []);
        """

        caminho_icone_processo = None
        prazo_icone_processo = time.monotonic() + 35
        while time.monotonic() < prazo_icone_processo:
            driver.switch_to.default_content()
            caminho_icone_processo = driver.execute_script(
                SCRIPT_BUSCAR_ICONE_PROCESSO, XPATH_ICONE_PROCESSO
            )
            if caminho_icone_processo is not None:
                break
            time.sleep(0.5)

        if caminho_icone_processo is None:
            raise TimeoutException(
                "O XPath do novo ícone não foi encontrado na janela principal "
                "do SEI nem em seus frames: " + XPATH_ICONE_PROCESSO
            )

        # Entra no mesmo frame identificado na pesquisa. Suporta iframe e frame.
        driver.switch_to.default_content()
        for indice in caminho_icone_processo:
            frames_disponiveis = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
            if indice >= len(frames_disponiveis):
                raise RuntimeError("A estrutura de frames mudou durante a busca do ícone.")
            driver.switch_to.frame(frames_disponiveis[indice])

        icone_processo = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, XPATH_ICONE_PROCESSO))
        )

        # O XPath termina em img; a ação costuma estar associada ao link pai.
        link_icone = driver.execute_script(
            "return arguments[0].closest('a') || arguments[0];",
            icone_processo
        )
        print("108 - Ícone localizado no frame:", caminho_icone_processo)
        print("Título do ícone:", icone_processo.get_attribute("title"))
        print("Título do link:", link_icone.get_attribute("title"))

        try:
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center',inline:'center'});",
                link_icone
            )
        except WebDriverException:
            pass

        try:
            link_icone.click()
            print("109 - Clique realizado no link do ícone.")
        except WebDriverException:
            driver.execute_script("arguments[0].click();", link_icone)
            print("109 - Clique realizado via JavaScript no link do ícone.")

        driver.switch_to.default_content()
        print("110 - Nova etapa concluída; janela principal do SEI preservada.")


        # ============================================================
        # 111. CLICAR NO CAMPO SELECT APÓS O ÍCONE DO PROCESSO
        #      Não altera nem seleciona qualquer opção da lista.
        # ============================================================

        XPATH_SELECT_PROCESSO = "/html/body/div[1]/div/div/form/div[2]/select"
        print("111 - Procurando o campo select após o clique no ícone...")

        # Reutiliza a busca em frames já existente no código anterior.
        caminho_select_processo = None
        prazo_select_processo = time.monotonic() + 35
        while time.monotonic() < prazo_select_processo:
            driver.switch_to.default_content()
            caminho_select_processo = driver.execute_script(
                SCRIPT_BUSCAR_ICONE_PROCESSO, XPATH_SELECT_PROCESSO
            )
            if caminho_select_processo is not None:
                break
            time.sleep(0.5)

        if caminho_select_processo is None:
            raise TimeoutException(
                "Campo select não encontrado na janela do SEI nem em seus frames: "
                + XPATH_SELECT_PROCESSO
            )

        driver.switch_to.default_content()
        for indice in caminho_select_processo:
            frames_disponiveis = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
            if indice >= len(frames_disponiveis):
                raise RuntimeError("A estrutura de frames mudou durante a busca do select.")
            driver.switch_to.frame(frames_disponiveis[indice])

        campo_select_processo = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, XPATH_SELECT_PROCESSO))
        )

        if campo_select_processo.tag_name.lower() != "select":
            raise RuntimeError("O XPath fornecido não corresponde a um campo select.")

        if not campo_select_processo.is_enabled():
            raise RuntimeError("O campo select foi encontrado, porém está desabilitado.")

        try:
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center',inline:'center'});",
                campo_select_processo
            )
        except WebDriverException:
            pass

        try:
            WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, XPATH_SELECT_PROCESSO))
            )
            campo_select_processo.click()
            print("112 - Clique realizado no campo select.")
        except (TimeoutException, WebDriverException) as erro_select:
            print("Clique convencional indisponível:", erro_select)
            driver.execute_script("arguments[0].click();", campo_select_processo)
            print("112 - Clique enviado ao campo select via JavaScript.")

        print("113 - Campo select acionado; nenhuma opção foi selecionada.")


        # ============================================================
        # 114. SELECIONAR BLOCO DE TESTE PARA AUTOMAÇÃO
        #      Valor exato da opção: 3767381
        # ============================================================

        from selenium.webdriver.support.ui import Select

        VALOR_BLOCO_AUTOMACAO = "3767381"
        print("114 - Aguardando opção do bloco:", VALOR_BLOCO_AUTOMACAO)

        # Permanecemos no mesmo contexto (janela e frame) do campo select.
        WebDriverWait(driver, 30).until(
            lambda d: any(
                opcao.get_attribute("value") == VALOR_BLOCO_AUTOMACAO
                for opcao in d.find_elements(
                    By.XPATH, XPATH_SELECT_PROCESSO + "/option"
                )
            )
        )

        campo_select_processo = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, XPATH_SELECT_PROCESSO))
        )

        Select(campo_select_processo).select_by_value(VALOR_BLOCO_AUTOMACAO)

        # A seleção pode disparar um refresh do formulário no SEI.
        # NÃO reutilizar campo_select_processo após select_by_value(): fica stale.
        print("115 - Seleção do bloco enviada ao SEI:", VALOR_BLOCO_AUTOMACAO)

        # Confere apenas se o select ainda existir após a atualização, sempre
        # buscando um elemento NOVO. Se a página tiver navegado, não repete
        # a seleção para evitar disparar uma segunda ação.
        valor_selecionado = None
        texto_selecionado = None
        formulario_atualizado = False
        prazo_confirmacao = time.monotonic() + 10

        while time.monotonic() < prazo_confirmacao:
            try:
                novos_campos = driver.find_elements(By.XPATH, XPATH_SELECT_PROCESSO)
                if not novos_campos:
                    formulario_atualizado = True
                    break

                campo_atual = novos_campos[0]
                valor_atual = campo_atual.get_attribute("value")
                if valor_atual == VALOR_BLOCO_AUTOMACAO:
                    valor_selecionado = valor_atual
                    try:
                        texto_selecionado = Select(campo_atual).first_selected_option.text.strip()
                    except (StaleElementReferenceException, WebDriverException):
                        texto_selecionado = None
                    break
            except (StaleElementReferenceException, WebDriverException):
                # O DOM pode ser substituído durante a atualização.
                formulario_atualizado = True
            time.sleep(0.5)

        if valor_selecionado == VALOR_BLOCO_AUTOMACAO:
            print("116 - Bloco confirmado:", valor_selecionado, texto_selecionado or "")
        else:
            print(
                "116 - A seleção foi enviada, mas o campo foi atualizado ou "
                "não permaneceu disponível para confirmação. "
                "Confira o resultado na tela do SEI antes de prosseguir."
            )

        # Volta ao documento principal para as próximas etapas.
        driver.switch_to.default_content()


        # ============================================================
        # 117. ÚLTIMA ETAPA: CLICAR NO BOTÃO "INCLUIR"
        # XPath informado: /html/body/div[1]/div/div/form/div[1]/button[1]
        # A busca é feita novamente após o refresh provocado pelo select.
        # ============================================================

        XPATH_BOTAO_INCLUIR = "/html/body/div[1]/div/div/form/div[1]/button[1]"
        print("117 - Aguardando o botão INCLUIR ficar disponível...")

        # A seleção do bloco pode atualizar o DOM; por isso não reutilizamos
        # nenhuma referência a elementos das etapas anteriores.
        def localizar_incluir_em_frames(profundidade=0):
            if profundidade > 5:
                return None
            try:
                botoes = driver.find_elements(By.XPATH, XPATH_BOTAO_INCLUIR)
                for botao in botoes:
                    try:
                        if botao.is_displayed() and botao.is_enabled():
                            return botao
                    except (StaleElementReferenceException, WebDriverException):
                        continue

                frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
                for indice in range(len(frames)):
                    try:
                        # Reobtém os frames para evitar stale durante atualizações.
                        frames_atuais = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
                        if indice >= len(frames_atuais):
                            break
                        driver.switch_to.frame(frames_atuais[indice])
                        encontrado = localizar_incluir_em_frames(profundidade + 1)
                        if encontrado is not None:
                            return encontrado  # mantém o contexto do frame correto
                        driver.switch_to.parent_frame()
                    except (StaleElementReferenceException, WebDriverException):
                        try:
                            driver.switch_to.parent_frame()
                        except WebDriverException:
                            pass
            except WebDriverException:
                pass
            return None

        janela_incluir = driver.current_window_handle
        botao_incluir = None
        fim_espera_incluir = time.monotonic() + 30

        while time.monotonic() < fim_espera_incluir:
            # Procura primeiro na janela onde o bloco acabou de ser selecionado.
            handles_incluir = driver.window_handles
            ordem_incluir = ([janela_incluir] if janela_incluir in handles_incluir else []) + [
                h for h in handles_incluir if h != janela_incluir
            ]
            for handle_incluir in ordem_incluir:
                try:
                    driver.switch_to.window(handle_incluir)
                    driver.switch_to.default_content()
                    botao_incluir = localizar_incluir_em_frames()
                    if botao_incluir is not None:
                        break
                except (StaleElementReferenceException, WebDriverException):
                    continue
            if botao_incluir is not None:
                break
            time.sleep(0.5)

        if botao_incluir is None:
            raise TimeoutException(
                "Botão INCLUIR não foi encontrado ou não ficou habilitado em 30 segundos: "
                + XPATH_BOTAO_INCLUIR
            )

        print("118 - Botão INCLUIR encontrado. Enviando clique único...")
        try:
            botao_incluir.click()
        except (StaleElementReferenceException, WebDriverException) as erro_incluir:
            # Se houve erro após um possível envio do clique, não repetir
            # automaticamente: poderia incluir o documento duas vezes.
            raise RuntimeError(
                "O clique em INCLUIR não foi confirmado pelo Selenium. "
                "Confira a tela antes de repetir a operação, para evitar inclusão duplicada."
            ) from erro_incluir

        print("119 - Clique no botão INCLUIR executado; confira a inclusão no SEI.")

        # O clique final foi enviado. Nao se presume comprovacao de persistencia.
        escrever_log(linha_excel, processo_log, ap_log, "CLIQUE_INCLUIR_ENVIADO", "Conferir inclusao efetiva no SEI")
        print(f"FIM DA LINHA {numero_linha}/{TOTAL}: clique INCLUIR enviado")

        # Aguarda a conclusao das transicoes de tela antes de iniciar outro registro.
        # Nao repete o clique INCLUIR nem autentica novamente.
        if numero_linha < TOTAL:
            print(f"Aguardando {TEMPO_ESPERA_ENTRE_PROCESSOS} segundos antes da proxima linha...")
            time.sleep(TEMPO_ESPERA_ENTRE_PROCESSOS)

        driver.switch_to.window(JANELA_SEI_PRINCIPAL)
        driver.switch_to.default_content()
        # Reutiliza o mesmo campo Pesquisa Rapida para a proxima linha.
        # Evita abrir novamente portal/URL e repetir o CPF e senha.
        if numero_linha < TOTAL:
            WebDriverWait(driver, 40).until(
                EC.presence_of_element_located((By.ID, "txtPesquisaRapida"))
            )

    except Exception as erro:
        print(f"ERRO na linha {linha_excel}, processo {processo_log}: {erro}")
        escrever_log(linha_excel, processo_log, ap_log, "ERRO_PAROU", f"{type(erro).__name__}: {erro}")
        # Pode ter ocorrido um salvamento ou clique antes da excecao.
        # Interrompe para evitar inclusoes duplicadas ou seguir na tela errada.
        raise RuntimeError(
            f"Processamento interrompido na linha {linha_excel}. "
            "Confira no SEI antes de executar novamente. "
            f"Log: {ARQUIVO_LOG}"
        ) from erro

print("\nPROCESSAMENTO TERMINADO. Log:", ARQUIVO_LOG)


1 - Página de login / autenticação aberta
Verificando se existe sessão autenticada...
Sessão não encontrada.
Login será realizado.
2 - CPF preenchido
3 - Botão continuar clicado
4 - Senha preenchida
5 - Login enviado
6 - Aguardando 20 segundos para carregar a aplicação...
7 - Procurando o elemento...
8 - NOW-CARD encontrado
9 - DIV interna encontrada
10 - CLIQUE FINAL REALIZADO COM SUCESSO!

11 - Aguardando janela / aba do SEI...
Verificando: Login Minha Área SP - Plataforma SP | https://minhaarea.sp.gov.br/plataformasp
Verificando: Página inicial | Minha Área SP.GOV.BR | ServiceNow | https://plataforma.sp.gov.br/x/mdpdd/minha-area-2-0/minha-area-2-0
Verificando: SEI - Controle de Processos | https://sei.sp.gov.br/sei/controlador.php?acao=procedimento_controlar&acao_origem=principal&acao_retorno=principal&inicializando=1&infra_sistema=100000100&infra_unidade_atual=110000700&infra_hash=e4f2192165d3ec9e8d0a32475e0e9c2a61612229d362aa55d407da2d2538e3b2

12 - Aba do SEI localizada.
Título: 